
# WHO PEN — Production-Grade Clinical RAG Pipeline — **v7 Production + GUI**

Complete Google Colab notebook for the 644-page WHO PEN 2018 manual.

**Architecture**

`Source Governance → Layout-Aware Parsing → Canonical Provenance → Correct Module Metadata → Structured Tables → L3/L4 Hierarchical Chunking → BGE-M3 + BM25 → RRF → BGE Cross-Encoder Reranking → Parent/Table Expansion → Corrective Retrieval → Grounded Generation → Citation/Numeric Verification → Entailment Verification → Evaluation → Gradio Clinical GUI`

### v7 fixes
- Can resume from the validated v6 checkpoint.
- `content_role` is advisory instead of a sole hard filter.
- RAGAS/LangChain compatibility is pinned.
- Cell 48 is resumable and rate-limit safe.
- Missing evaluation is **PENDING**, never silently PASS.
- Persisted manifests are secret-safe.
- Checkpoint saving is verified before replacing the previous copy.
- Adds a polished Gradio Blocks GUI with subtle animations, progress, sources, evidence, safety status, and developer trace.

> Clinical-reference assistant only. Final release still requires the agreed held-out / clinician-reviewed acceptance gates.


## Runtime guide — which cells are expected to take time?

There is **no model training** in this notebook. BGE-M3 and the cross-encoder are used for inference only.

Typical Colab T4/L4 behavior:

| Cell / stage | Typical runtime | Why it takes time |
|---|---:|---|
| Cell 1 — Install dependencies | 2–8 min | Large Python/ML packages |
| Cell 4 — Download PDF | <1–3 min | Downloads ~26 MB source PDF |
| Cell 6 — Raw PDF audit | 1–4 min | Reads all 644 pages |
| **Cell 8 — Docling full parse** | **15–60+ min** | **OCR + layout + tables + pictures across all 644 pages** |
| Cell 15 — Canonical element ledger | 1–6 min | Iterates every parsed document item |
| Cell 16 — Table extraction | 1–10 min | Reconstructs structured tables / Markdown / HTML |
| Cells 21–23 — Hierarchical chunking | 1–8 min total | Tokenizes and builds L3/L4 hierarchy |
| Cell 25 — Load BGE-M3 | 1–4 min | Downloads/loads embedding model on first run |
| **Cell 25B — Embedding batch benchmark** | **1–5 min** | Tests several batch sizes and measures VRAM/throughput |
| **Cell 26 — Full embedding** | **5–30+ min** | Embeds every L4 leaf; depends on number of leaves and GPU |
| Cell 27 — Qdrant indexing | 1–6 min | Writes all vectors + metadata |
| Cell 28 — BM25 index | 1–5 min | Tokenizes all leaves for sparse retrieval |
| Cell 32 — Load reranker | 1–4 min | Downloads/loads cross-encoder |
| Cell 32B — Reranker batch benchmark | <1–4 min | Benchmarks safe online reranker batches |
| Cell 37 — Smoke tests | seconds–few min | Dense + BM25 + reranking |
| Cell 45 — Bootstrap retrieval evaluation | 2–15 min | Runs retrieval repeatedly over the regression set |
| **Cell 48 — RAGAS evaluation** | **variable / potentially long** | Multiple external LLM judge calls per question |

The largest one-time bottleneck is normally **Cell 8 (Docling parsing)**. After caching, later runs skip the expensive parse. The other major one-time stage is **Cell 26 (full-corpus embedding)**, which is also cached.

### Auto-resume rule
- **Checkpoint exists:** restore it and reuse only artifacts that pass each stage's validation/fingerprint.
- **Checkpoint absent:** run the complete pipeline normally from the source PDF.
- The notebook does **not** depend on any v5 checkpoint.


## Colab environment note — v6 Clean

On the **first run**, Cell 1 may restart the Python kernel once after installing the pinned numerical stack. After Colab reconnects, run Cell 1 again and continue.

v6 uses a new versioned workspace, so v5 stale parents/leaves/embeddings/Qdrant/BM25 cannot contaminate this run. A later helper can safely reuse only the old source PDF and Docling JSON cache.


In [ ]:
# Cell 1 — Deterministic Colab environment bootstrap
# IMPORTANT:
# - Pins NumPy to a known-stable Python 3.12 build.
# - FIRST run: installs packages, validates them in a fresh process, then
#   restarts the Colab kernel once to prevent mixed NumPy C-extension state.
# - After Colab reconnects, run the notebook again from the top.
# - Subsequent runs only perform a health check.

import os
import sys
import signal
import subprocess
from pathlib import Path

ENV_SENTINEL = Path("/content/.who_pen_env_v7_ready")

NUMPY_VERSION = "2.2.6"
SCIPY_VERSION = "1.16.1"

packages = [
    f"numpy=={NUMPY_VERSION}",
    f"scipy=={SCIPY_VERSION}",
    "docling>=2.60,<3",
    "qdrant-client>=1.16,<2",
    "sentence-transformers>=5,<6",
    "transformers>=4.55,<5",
    "accelerate>=1.10,<2",
    "rank-bm25>=0.2.2",
    "pymupdf>=1.26",
    "gdown>=5.2",
    "pandas>=2.2,<3",
    "pyarrow>=18",
    "scikit-learn>=1.5,<2",
    "rapidfuzz>=3.10",
    "pydantic>=2.10,<3",
    "tenacity>=9",
    "requests>=2.32",
    "openai>=1.60",
    "datasets>=3,<5",
    "ragas==0.4.3",
    "langchain-community>=0.4,<0.5",
    "langchain-google-vertexai>=3,<4",
    "gradio>=5,<6",
    "tabulate>=0.9",
    "joblib>=1.4",
]


def patch_ragas_vertexai_imports():
    """
    RAGAS 0.4.3 still imports VertexAI from removed langchain-community paths.
    Patch those two import lines to the maintained langchain-google-vertexai package.
    Safe to run repeatedly; no-op once patched.
    """
    import site

    candidates = []
    for root in site.getsitepackages():
        candidates.append(Path(root) / "ragas" / "llms" / "base.py")
    candidates.append(Path(site.getusersitepackages()) / "ragas" / "llms" / "base.py")

    base_py = next((p for p in candidates if p.exists()), None)
    if base_py is None:
        raise RuntimeError("Could not locate ragas/llms/base.py for compatibility patch.")

    text = base_py.read_text(encoding="utf-8")
    original = text

    text = text.replace(
        "from langchain_community.chat_models.vertexai import ChatVertexAI",
        "from langchain_google_vertexai import ChatVertexAI",
    )
    text = text.replace(
        "from langchain_community.llms import VertexAI",
        "from langchain_google_vertexai import VertexAI",
    )

    if text != original:
        base_py.write_text(text, encoding="utf-8")
        print("✓ Patched RAGAS 0.4.3 VertexAI compatibility imports.")
    else:
        print("✓ RAGAS VertexAI compatibility patch already applied.")

def fresh_process_health_check():
    check_code = '''
import sys
import numpy
import scipy
import pandas
import ragas
import gradio
from langchain_google_vertexai import ChatVertexAI, VertexAI
from docling.document_converter import DocumentConverter, PdfFormatOption

print("Python :", sys.version.split()[0])
print("NumPy  :", numpy.__version__)
print("SciPy  :", scipy.__version__)
print("Pandas :", pandas.__version__)
print("RAGAS  :", ragas.__version__)
print("Gradio :", gradio.__version__)
print("Docling import: OK")
print("RAGAS/LangChain compatibility import: OK")

assert numpy.__version__ == "2.2.6", numpy.__version__
'''
    subprocess.check_call([sys.executable, "-c", check_code])

if not ENV_SENTINEL.exists():
    print("=" * 80)
    print("FIRST-RUN ENVIRONMENT SETUP")
    print("=" * 80)

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--upgrade",
            "--force-reinstall",
            *packages,
        ]
    )

    patch_ragas_vertexai_imports()

    print("\nFresh-process dependency health check...")
    fresh_process_health_check()

    ENV_SENTINEL.write_text("ready", encoding="utf-8")

    print("\n" + "=" * 80)
    print("INSTALLATION VERIFIED")
    print("=" * 80)
    print("The kernel will restart ONCE now.")
    print("After Colab reconnects: run the notebook again from the top.")
    print("=" * 80)

    sys.stdout.flush()
    sys.stderr.flush()
    os.kill(os.getpid(), signal.SIGKILL)

else:
    print("=" * 80)
    print("ENVIRONMENT HEALTH CHECK")
    print("=" * 80)

    # Keep the live Colab kernel clean here. Heavy/native Docling imports are
    # validated in a short-lived subprocess and only loaded in the main kernel
    # immediately before parsing.
    patch_ragas_vertexai_imports()
    fresh_process_health_check()

    import numpy as _np
    import scipy as _scipy
    import pandas as _pd

    print("Python :", sys.version.split()[0])
    print("NumPy  :", _np.__version__)
    print("SciPy  :", _scipy.__version__)
    print("Pandas :", _pd.__version__)
    print("Docling subprocess import: OK")

    if _np.__version__ != NUMPY_VERSION:
        raise RuntimeError(
            f"Expected NumPy {NUMPY_VERSION}, found {_np.__version__}. "
            "Delete /content/.who_pen_env_v7_ready and rerun Cell 1."
        )

    print("✓ Environment is consistent. Continue to Cell 2.")


In [ ]:
# Cell 2 — Imports, runtime and reproducibility

import os
import re
import gc
import json
import math
import time
import uuid
import hashlib
import logging
import platform
from pathlib import Path
from dataclasses import dataclass, asdict, field
from collections import Counter, defaultdict
from typing import Any, Optional, Literal

import numpy as np
import pandas as pd
import torch
import fitz
import joblib
import requests
from tqdm.auto import tqdm
from pydantic import BaseModel, Field
from tenacity import retry, stop_after_attempt, wait_exponential

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2B — Stage timing + GPU memory utilities

from contextlib import contextmanager

STAGE_TIMINGS = {}

@contextmanager
def stage_timer(stage_name: str):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        STAGE_TIMINGS[stage_name] = elapsed
        print(
            f"\n⏱ {stage_name}: "
            f"{elapsed/60:.2f} min ({elapsed:.1f} sec)"
        )

def gpu_memory_report(label: str = "GPU"):
    if not torch.cuda.is_available():
        print(f"{label}: CUDA not available")
        return {
            "total_gb": 0.0,
            "allocated_gb": 0.0,
            "reserved_gb": 0.0,
            "peak_gb": 0.0,
            "peak_pct": 0.0,
        }

    total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
    peak_pct = (peak / total * 100.0) if total else 0.0

    report = {
        "total_gb": total,
        "allocated_gb": allocated,
        "reserved_gb": reserved,
        "peak_gb": peak,
        "peak_pct": peak_pct,
    }

    print("=" * 70)
    print(label)
    print("=" * 70)
    print(f"Total VRAM        : {total:.2f} GB")
    print(f"Current allocated : {allocated:.2f} GB")
    print(f"Current reserved  : {reserved:.2f} GB")
    print(f"Peak allocated    : {peak:.2f} GB")
    print(f"Peak utilization  : {peak_pct:.1f}%")
    print("=" * 70)

    return report

In [ ]:
# Cell 3 — Global configuration (v6 clean workspace)

@dataclass
class Config:
    # Source
    drive_url: str = os.getenv("WHO_PEN_DRIVE_URL", "")
    expected_pages: int = 644
    document_id: str = "who_pen_2018_training_modules"
    document_title: str = "Package of Essential Noncommunicable (PEN) disease and healthy lifestyle interventions"
    publisher: str = "World Health Organization"
    edition: str = "First Edition"
    publication_year: int = 2018
    language: str = "en"
    jurisdiction: str = "WHO South-East Asia Region"
    source_status: str = "training_material"

    # Versioned local workspace — intentionally separate from v5
    root: Path = Path("/content/who_pen_medical_rag_v7_gui")
    pdf_name: str = "who_pen_full.pdf"

    # Chunking
    parent_max_tokens: int = 760
    parent_overlap_ratio: float = 0.12
    leaf_max_tokens: int = 220
    leaf_overlap_ratio: float = 0.12

    # Models
    embedding_model: str = "BAAI/bge-m3"
    reranker_model: str = "BAAI/bge-reranker-v2-m3"
    embedding_batch_size: int = 0
    reranker_batch_size: int = 0

    # Retrieval
    dense_k: int = 60
    bm25_k: int = 60
    fusion_k: int = 50
    rerank_k: int = 30
    final_leaf_k: int = 10
    final_parent_k: int = 5
    max_leaves_per_parent: int = 3
    rrf_constant: int = 60
    max_context_tokens: int = 6000
    corrective_retrieval_loops: int = 2

    # Reranker sufficiency score. This sigmoid(logit) value is NOT a calibrated
    # probability. Keep threshold low until calibrated on the human gold set.
    min_top_reranker_prob: float = 0.02

    # LLM: OpenAI-compatible API
    llm_base_url: str = os.getenv("LLM_BASE_URL", "https://api.openai.com/v1")
    llm_api_key: str = os.getenv("LLM_API_KEY", "")
    llm_model: str = os.getenv("LLM_MODEL", "")
    verifier_model: str = os.getenv("VERIFIER_MODEL", "")
    request_timeout: int = 120

    hard_filter_confidence: float = 0.92
    allowed_filter_fields: tuple = ("theme_id", "module_id", "content_role")

CFG = Config()

CFG.root.mkdir(parents=True, exist_ok=True)
for sub in [
    "source", "cache", "canonical", "chunks", "index",
    "eval", "logs", "visual_qa"
]:
    (CFG.root / sub).mkdir(parents=True, exist_ok=True)

PDF_PATH = CFG.root / "source" / CFG.pdf_name
DOCLING_JSON = CFG.root / "cache" / "docling_document.json"
ELEMENTS_PATH = CFG.root / "canonical" / "elements.parquet"
TABLES_PATH = CFG.root / "canonical" / "tables.jsonl"
MODULE_RANGES_PATH = CFG.root / "canonical" / "module_ranges.parquet"
PAGE_META_PATH = CFG.root / "canonical" / "page_meta.json"
PARENTS_PATH = CFG.root / "chunks" / "parents.parquet"
LEAVES_PATH = CFG.root / "chunks" / "leaves.parquet"
EMBEDDINGS_PATH = CFG.root / "index" / "leaf_embeddings.npy"
EMBEDDINGS_META_PATH = CFG.root / "index" / "leaf_embeddings.meta.json"
BM25_PATH = CFG.root / "index" / "bm25.joblib"
QDRANT_DIR = CFG.root / "index" / "qdrant"
QDRANT_META_PATH = CFG.root / "index" / "qdrant.meta.json"
MANIFEST_PATH = CFG.root / "source" / "source_manifest.json"

print(asdict(CFG))


In [ ]:
# Cell 3B — Adaptive inference batch sizing from available GPU VRAM

def auto_batch_sizes(device: str):
    # Conservative production defaults for BGE-M3 + BGE-reranker-v2-m3.
    if device != "cuda" or not torch.cuda.is_available():
        return 4, 2

    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    if total_gb >= 70:
        return 64, 24
    elif total_gb >= 38:
        return 32, 16
    elif total_gb >= 22:
        return 24, 12
    elif total_gb >= 14:
        # Colab T4 / L4-class safe defaults.
        return 16, 8
    elif total_gb >= 10:
        return 8, 4
    else:
        return 4, 2

auto_embed_bs, auto_rerank_bs = auto_batch_sizes(DEVICE)

if CFG.embedding_batch_size <= 0:
    CFG.embedding_batch_size = auto_embed_bs

if CFG.reranker_batch_size <= 0:
    CFG.reranker_batch_size = auto_rerank_bs

print("Adaptive embedding batch size :", CFG.embedding_batch_size)
print("Adaptive reranker batch size  :", CFG.reranker_batch_size)

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print("GPU VRAM (GB):", round(total_gb, 2))

In [ ]:
# Cell 3C — OPTIONAL checkpoint auto-resume
# Priority: v7 checkpoint -> validated v6 checkpoint -> fresh rebuild.

from google.colab import drive
import shutil

drive.mount('/content/drive', force_remount=False)

DRIVE_PROJECT_ROOT = Path(
    os.getenv(
        "WHO_PEN_DRIVE_ROOT",
        "/content/drive/MyDrive/WHO_PEN_RAG",
    )
)

CHECKPOINT_CANDIDATES = [
    DRIVE_PROJECT_ROOT / "who_pen_medical_rag_v7_gui_checkpoint",
    DRIVE_PROJECT_ROOT / "who_pen_medical_rag_v6_clean_checkpoint",
]

CHECKPOINT_LOADED = False
CHECKPOINT_SOURCE = None

for candidate in CHECKPOINT_CANDIDATES:
    if candidate.exists() and candidate.is_dir():
        CHECKPOINT_SOURCE = candidate
        break

if CHECKPOINT_SOURCE is not None:
    print('✓ Checkpoint detected:')
    print(' ', CHECKPOINT_SOURCE)

    if CFG.root.exists():
        shutil.rmtree(CFG.root)

    shutil.copytree(CHECKPOINT_SOURCE, CFG.root)
    CHECKPOINT_LOADED = True

    print('✅ Checkpoint restored into the v7 workspace.')
    print('✅ Fingerprinted stages will independently validate/rebuild stale artifacts.')
else:
    print('ℹ️ No checkpoint found.')
    print('✅ Starting a normal fresh run from the source PDF.')

print('\nCheckpoint loaded:', CHECKPOINT_LOADED)
print('Checkpoint source:', CHECKPOINT_SOURCE)


In [ ]:
# Cell 4 — Robustly load the WHO PEN PDF
# Priority:
#   1) Existing local cache
#   2) Mounted Google Drive folder
#   3) Optional WHO_PEN_DRIVE_URL via gdown
#
# Public GitHub note:
# The source PDF is intentionally not committed to this repository.
# Put 9789290226666-eng.pdf inside WHO_PEN_DRIVE_ROOT or provide
# WHO_PEN_DRIVE_URL as an environment variable / Colab secret.

import shutil

SOURCE_FILENAME = "9789290226666-eng.pdf"

DRIVE_PROJECT_ROOT = Path(
    os.getenv(
        "WHO_PEN_DRIVE_ROOT",
        "/content/drive/MyDrive/WHO_PEN_RAG",
    )
)

def ensure_pdf_available():
    PDF_PATH.parent.mkdir(parents=True, exist_ok=True)

    # A) Local cache
    if PDF_PATH.exists() and PDF_PATH.stat().st_size > 1_000_000:
        print("✓ Using cached local PDF:", PDF_PATH)
        return PDF_PATH

    # B) Mounted Drive
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)

        likely_paths = [
            DRIVE_PROJECT_ROOT / SOURCE_FILENAME,
            Path("/content/drive/MyDrive") / SOURCE_FILENAME,
        ]

        found = next((p for p in likely_paths if p.exists()), None)

        if found is not None:
            print("✓ Found source PDF:")
            print(" ", found)
            shutil.copy2(found, PDF_PATH)
            print("✓ Copied to notebook workspace:")
            print(" ", PDF_PATH)
            return PDF_PATH

    except Exception as e:
        print("Drive mount/search skipped:", repr(e))

    # C) Optional shared URL
    if CFG.drive_url:
        import gdown

        print("Trying WHO_PEN_DRIVE_URL with gdown...")
        out = gdown.download(
            url=CFG.drive_url,
            output=str(PDF_PATH),
            fuzzy=True,
            quiet=False,
        )
        if out and PDF_PATH.exists():
            return PDF_PATH

    raise FileNotFoundError(
        "WHO PEN PDF could not be obtained.\n"
        f"Expected filename: {SOURCE_FILENAME}\n"
        f"Recommended Drive folder: {DRIVE_PROJECT_ROOT}\n"
        "Either place the source PDF there or set WHO_PEN_DRIVE_URL."
    )

_stage_t0 = time.perf_counter()

PDF_PATH = ensure_pdf_available()

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF missing after acquisition step: {PDF_PATH}")

if PDF_PATH.stat().st_size < 1_000_000:
    raise ValueError(
        f"Downloaded/copied file is suspiciously small: "
        f"{PDF_PATH.stat().st_size / 1024:.1f} KB"
    )

with PDF_PATH.open("rb") as f:
    magic = f.read(5)

if magic != b"%PDF-":
    raise ValueError(
        f"The acquired file is not a valid PDF. First bytes: {magic!r}"
    )

print("\n" + "=" * 80)
print("SOURCE PDF READY")
print("=" * 80)
print("Path    :", PDF_PATH)
print("Size MB :", round(PDF_PATH.stat().st_size / 1024**2, 2))
print("=" * 80)

print(
    f"\n⏱ 04 Load PDF: "
    f"{(time.perf_counter() - _stage_t0)/60:.2f} min"
)


In [ ]:
# Cell 5 — Source governance, hash and page validation

def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found at {PDF_PATH}. Run Cell 4 first. "
        "Cell 4 mounts Google Drive, finds 9789290226666-eng.pdf, "
        "and copies it into the workspace."
    )

pdf = fitz.open(PDF_PATH)
page_count = len(pdf)
pdf_sha256 = sha256_file(PDF_PATH)

if page_count != CFG.expected_pages:
    raise ValueError(
        f"Expected {CFG.expected_pages} pages but found {page_count}. "
        "Stop here: wrong or changed source file."
    )

source_registry = {
    "document_id": CFG.document_id,
    "document_title": CFG.document_title,
    "publisher": CFG.publisher,
    "edition": CFG.edition,
    "publication_year": CFG.publication_year,
    "language": CFG.language,
    "jurisdiction": CFG.jurisdiction,
    "source_status": CFG.source_status,
    "source_url": CFG.drive_url,
    "source_sha256": pdf_sha256,
    "page_count": page_count,
    "ingested_at_utc": pd.Timestamp.utcnow().isoformat(),
}

MANIFEST_PATH.write_text(
    json.dumps(source_registry, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(source_registry, indent=2, ensure_ascii=False))

In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 6 — Fast raw-PDF audit before layout parsing

def normalize_line_for_frequency(s: str) -> str:
    s = re.sub(r"\s+", " ", s.strip())
    s = re.sub(r"\b\d{1,4}\b", "<NUM>", s)
    return s.lower()

page_audit = []
short_line_pages = defaultdict(set)
raw_page_texts = {}

for page_no in tqdm(range(1, page_count + 1), desc="Auditing pages"):
    page = pdf[page_no - 1]
    text = page.get_text("text") or ""
    raw_page_texts[page_no] = text

    image_count = len(page.get_images(full=True))
    blocks = page.get_text("blocks")
    lines = [x.strip() for x in text.splitlines() if x.strip()]

    for line in lines:
        norm = normalize_line_for_frequency(line)
        if 3 <= len(norm) <= 120:
            short_line_pages[norm].add(page_no)

    page_audit.append({
        "page": page_no,
        "chars": len(text),
        "words": len(text.split()),
        "images": image_count,
        "blocks": len(blocks),
        "text_sparse": len(text.strip()) < 80,
    })

audit_df = pd.DataFrame(page_audit)

repeated_candidates = {
    line: pages
    for line, pages in short_line_pages.items()
    if len(pages) >= max(12, int(page_count * 0.08))
}

print(audit_df.describe(include="all"))
print("\nText-sparse pages:", int(audit_df["text_sparse"].sum()))
print("Repeated short-line candidates:", len(repeated_candidates))
print(f'\n⏱ 06 Raw PDF audit: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')

# Release the native PyMuPDF document before Docling initializes its PDF stack.
try:
    pdf.close()
except Exception:
    pass
try:
    del pdf
except Exception:
    pass

import gc as _gc_after_pdf_audit
_gc_after_pdf_audit.collect()
print("✓ Raw-PDF audit complete; PyMuPDF handle released before Docling.")


In [ ]:
# Cell 7 — Colab-stable Docling configuration (lightweight; no model init yet)
#
# Why this version is safer:
# - releases native resources before Docling model initialization
# - limits native CPU thread pools
# - uses CPU for Docling/TableFormer to avoid a fragile CUDA/native initialization
#   during the ingestion stage
# - DOES NOT instantiate DocumentConverter here; Cell 8 creates it only if the
#   Docling JSON cache is actually missing

import os
import gc

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Clean up any stale converter/native objects from a previous partial run.
for _name in ["converter", "conv_result", "pipeline_options"]:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass

gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

# NumPy ABI preflight
try:
    import numpy as _np_preflight
    from numpy._core import umath as _umath_preflight
    if _np_preflight.__version__ != "2.2.6":
        raise RuntimeError(
            f"Unexpected NumPy version {_np_preflight.__version__}. "
            "Run Cell 1 and allow the one-time runtime restart."
        )
except Exception as e:
    raise RuntimeError(
        "NumPy is not healthy in this Colab kernel. "
        "Run Cell 1, allow the restart, then continue."
    ) from e

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.accelerator_options import AcceleratorOptions, AcceleratorDevice
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()

# Native text + high-quality table structure, without OCR / picture inference.
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
pipeline_options.table_structure_options.do_cell_matching = True
pipeline_options.generate_picture_images = False
pipeline_options.do_picture_classification = False

# Disable optional image/enrichment outputs when supported by the installed Docling version.
for _attr, _value in [
    ("generate_page_images", False),
    ("generate_parsed_pages", False),
    ("do_picture_description", False),
    ("do_formula_enrichment", False),
    ("do_code_enrichment", False),
]:
    if hasattr(pipeline_options, _attr):
        setattr(pipeline_options, _attr, _value)

# Stability-first ingestion. The T4 remains available later for BGE-M3 and reranking.
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=2,
    device=AcceleratorDevice.CPU,
)

# Do NOT create DocumentConverter in this cell.
converter = None

# Lightweight RAM report
try:
    import psutil
    _vm = psutil.virtual_memory()
    print(
        f"RAM available before Docling parse: "
        f"{_vm.available / (1024**3):.2f} GB / "
        f"{_vm.total / (1024**3):.2f} GB"
    )
except Exception:
    pass

print("✓ NumPy preflight passed:", _np_preflight.__version__)
print("✓ Docling configured: native text + ACCURATE tables.")
print("✓ Docling ingestion device: CPU (stability-first).")
print("✓ DocumentConverter initialization deferred to Cell 8.")


In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()

# Cell 8 — Parse once OR load cached DoclingDocument (Colab-stable)
from docling_core.types.doc import DoclingDocument
import gc

_cache_ok = (
    DOCLING_JSON.exists()
    and DOCLING_JSON.stat().st_size > 1_000_000
)

if _cache_ok:
    print("Loading cached DoclingDocument:", DOCLING_JSON)
    dl_doc = DoclingDocument.load_from_json(DOCLING_JSON)
    print("✓ Cache hit — DocumentConverter/TableFormer were NOT initialized.")

else:
    print("No valid Docling JSON cache found.")
    print("Parsing full PDF. This is the expensive ingestion stage...")

    # Create the converter ONLY when a fresh parse is genuinely required.
    converter = DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options
            )
        },
    )

    start = time.time()

    conv_result = converter.convert(
        PDF_PATH,
        raises_on_error=True,
    )

    dl_doc = conv_result.document
    dl_doc.save_as_json(DOCLING_JSON)

    print(
        "Docling parse minutes:",
        round((time.time() - start) / 60, 2)
    )

    # Release the converter and native/model resources immediately after caching.
    try:
        if hasattr(converter, "initialized_pipelines"):
            converter.initialized_pipelines.clear()
    except Exception:
        pass

    try:
        del conv_result
    except Exception:
        pass

    try:
        del converter
    except Exception:
        pass

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

print("Docling pages:", dl_doc.num_pages())
print("Texts:", len(dl_doc.texts))
print("Tables:", len(dl_doc.tables))
print("Pictures:", len(dl_doc.pictures))

assert dl_doc.num_pages() == CFG.expected_pages, (
    f"Expected {CFG.expected_pages} pages, got {dl_doc.num_pages()}"
)

print(
    f"\n⏱ 08 Docling load/parse: "
    f"{(_stage_time.perf_counter() - _stage_t0)/60:.2f} min"
)


In [ ]:
# Cell 9 — Canonical parsing QA: coverage, provenance and structure

from docling_core.types.doc import TextItem, TableItem, PictureItem

qa_counts = Counter()
pages_with_provenance = set()
items_without_provenance = 0

for item, level in dl_doc.iterate_items(traverse_pictures=True):
    qa_counts[type(item).__name__] += 1
    prov = getattr(item, "prov", None) or []
    if prov:
        pages_with_provenance.update(p.page_no for p in prov)
    else:
        items_without_provenance += 1

qa_report = {
    "item_types": dict(qa_counts),
    "pages_with_provenance": len(pages_with_provenance),
    "page_coverage_ratio": len(pages_with_provenance) / page_count,
    "items_without_provenance": items_without_provenance,
    "tables": len(dl_doc.tables),
    "pictures": len(dl_doc.pictures),
}

print(json.dumps(qa_report, indent=2))

In [ ]:
# Cell 10 — Utilities: safe text normalization without changing clinical meaning

BAD_CONTROL_CHARS = {
    "\u00ad": "",      # soft hyphen
    "\ufffe": "",
    "\ufeff": "",
}

def normalize_medical_text(text: str) -> str:
    if not text:
        return ""
    out = text
    for src, dst in BAD_CONTROL_CHARS.items():
        out = out.replace(src, dst)

    # Join only obvious alphabetic line-break hyphenations.
    out = re.sub(r"(?<=[A-Za-z])[-‐]\s*\n\s*(?=[a-z])", "", out)

    # Preserve ≥ ≤ µ μ % ranges and dosage expressions.
    out = out.replace("–", "–").replace("—", "—")
    out = re.sub(r"[ \t]+", " ", out)
    out = re.sub(r"\n{3,}", "\n\n", out)
    return out.strip()

def safe_bbox(prov) -> Optional[dict]:
    if prov is None:
        return None
    bbox = getattr(prov, "bbox", None)
    if bbox is None:
        return None
    try:
        return bbox.model_dump()
    except Exception:
        return {
            k: getattr(bbox, k, None)
            for k in ("l", "t", "r", "b", "coord_origin")
        }

def label_name(item) -> str:
    label = getattr(item, "label", "")
    value = getattr(label, "value", str(label))
    return str(value).lower()

def content_layer_name(item) -> str:
    cl = getattr(item, "content_layer", "")
    return str(getattr(cl, "value", cl)).lower()

print(normalize_medical_text("SBP ≥140 mmHg and/or\nDBP ≥90 mmHg."))

In [ ]:
# Cell 11 — Discover authoritative module boundaries + the TRUE final Annex 1

MODULE_TITLES = {
    "1.1": "NCD burden and public health approaches",
    "1.2": "PHC approach to delivering essential NCD services",
    "2.1": "Addressing healthy lifestyle in primary health care",
    "2.2": "Brief interventions for tobacco cessation at the primary health care level",
    "2.3": "Brief interventions for harmful alcohol use at primary health care level",
    "2.4": "Providing brief intervention for a healthy diet at primary health care level",
    "2.5": "Promotion of physical activity in primary health care",
    "2.6": "Addressing overweight and obesity in primary health care",
    "2.7": "Understanding the health impacts of household air pollution at the primary health care level",
    "3.1": "Prevention and management of cardiovascular diseases in primary health care",
    "3.2": "Management of hypertension at primary health care",
    "3.3": "Management of type 2 diabetes in the primary health care level",
    "3.4": "Prevention of stroke in primary health care",
    "3.5": "Prevention of rheumatic heart disease (RHD)",
    "3.6": "Managing COPD and asthma",
    "3.7": "Early detection and referral of suspected oral cancers in primary health care",
    "3.8": "Assessment and referral of suspected breast cancer at primary health care",
    "3.9": "Assessment and referral of women with suspected cervical cancer at primary health care level",
    "4.1": "Addressing comorbidities of common mental disorders and NCDs",
    "4.2": "Palliative care in primary health care",
    "5.1": "Organizing NCD services through a team-based approach",
    "5.2": "Monitoring systems",
}

module_hits = []
for page_no, text in raw_page_texts.items():
    for module_id in re.findall(r"\bModule\s+([1-5]\.\d+)\b", text, flags=re.I):
        if module_id in MODULE_TITLES:
            module_hits.append({
                "module_id": module_id,
                "module_title": MODULE_TITLES[module_id],
                "start_page": int(page_no),
            })

hit_df = pd.DataFrame(module_hits)
if hit_df.empty:
    raise RuntimeError("No numbered WHO PEN modules were discovered.")

counts = hit_df.groupby("module_id").size()
ambiguous = counts[counts != 1]
if len(ambiguous):
    raise RuntimeError(f"Ambiguous module title-page discovery:\n{ambiguous}")

module_starts = hit_df.sort_values("start_page").to_dict("records")
found_ids = {x["module_id"] for x in module_starts}
expected_ids = set(MODULE_TITLES)
assert found_ids == expected_ids, (
    f"Module discovery mismatch. Missing={sorted(expected_ids-found_ids)} "
    f"Extra={sorted(found_ids-expected_ids)}"
)

# Critical fix: the document has other local 'Annex 1' occurrences. The final Annex
# must occur AFTER the final numbered module and contain its actual title.
last_module_start = max(x["start_page"] for x in module_starts)
annex_candidates = []
for page_no, text in raw_page_texts.items():
    if page_no <= last_module_start:
        continue
    if (
        re.search(r"\bAnnex\s+1\b", text, re.I)
        and re.search(r"\bMaking\s+training\s+effective\b", text, re.I)
    ):
        annex_candidates.append(int(page_no))

print("Modules discovered:", len(module_starts))
print("Final annex candidates:", annex_candidates)
assert annex_candidates == [629], (
    f"Expected the final 'Making training effective' Annex 1 at PDF page 629; "
    f"found {annex_candidates}. Stop because the source edition may differ."
)
annex_page = annex_candidates[0]

THEMES = {
    "theme_1": "Overview of the burden of NCDs and cost-effective public health interventions",
    "theme_2": "Health education and counselling on lifestyle risk factors for NCDs",
    "theme_3": "Approaches to NCD management in the PHC setting",
    "theme_4": "Delivering PEN services within the health facility",
}

def theme_for_module(module_id: str) -> tuple[str, str]:
    if module_id.startswith("1."):
        return "theme_1", THEMES["theme_1"]
    if module_id.startswith("2."):
        return "theme_2", THEMES["theme_2"]
    if module_id.startswith(("3.", "4.")):
        return "theme_3", THEMES["theme_3"]
    if module_id.startswith("5."):
        return "theme_4", THEMES["theme_4"]
    raise ValueError(f"Unknown numbered module: {module_id}")

module_ranges = [{
    "module_id": "front_matter",
    "module_title": "Front matter and scope of training modules",
    "start_page": 1,
    "end_page": module_starts[0]["start_page"] - 1,
    "theme_id": "front_matter",
    "theme_title": "Front matter / scope",
}]

for i, mod in enumerate(module_starts):
    next_start = (
        module_starts[i + 1]["start_page"]
        if i + 1 < len(module_starts)
        else annex_page
    )
    theme_id, theme_title = theme_for_module(mod["module_id"])
    module_ranges.append({
        **mod,
        "end_page": int(next_start) - 1,
        "theme_id": theme_id,
        "theme_title": theme_title,
    })

module_ranges.append({
    "module_id": "annex_1",
    "module_title": "Making training effective",
    "start_page": annex_page,
    "end_page": page_count,
    "theme_id": "annex",
    "theme_title": "Annex",
})

module_df = pd.DataFrame(module_ranges).sort_values("start_page").reset_index(drop=True)

# No overlaps or gaps.
assert int(module_df.iloc[0]["start_page"]) == 1
assert int(module_df.iloc[-1]["end_page"]) == page_count
for i in range(len(module_df) - 1):
    assert int(module_df.iloc[i]["end_page"]) + 1 == int(module_df.iloc[i+1]["start_page"])

print("✓ TRUE final Annex page:", annex_page)
print("✓ Module ranges are contiguous and non-overlapping.")
display(module_df[["module_id", "module_title", "start_page", "end_page", "theme_id"]])


In [ ]:
# Cell 12 — Assign page → theme/module lookup, validate, and persist

page_meta = {}
for row in module_ranges:
    for p in range(int(row["start_page"]), int(row["end_page"]) + 1):
        if p in page_meta:
            raise RuntimeError(f"Module metadata overlap at page {p}")
        page_meta[p] = {
            "theme_id": row["theme_id"],
            "theme_title": row["theme_title"],
            "module_id": row["module_id"],
            "module_title": row["module_title"],
        }

assert len(page_meta) == page_count

EXPECTED_PAGE_MODULES = {
    14: "front_matter",
    81: "2.1", 85: "2.1",
    127: "2.3", 129: "2.3",
    250: "3.2", 257: "3.2",
    601: "5.2", 624: "5.2",
    629: "annex_1", 631: "annex_1",
}
for p, expected in EXPECTED_PAGE_MODULES.items():
    actual = page_meta[p]["module_id"]
    assert actual == expected, f"Page {p}: expected {expected}, found {actual}"

module_df.to_parquet(MODULE_RANGES_PATH, index=False)
PAGE_META_PATH.write_text(
    json.dumps({str(k): v for k, v in page_meta.items()}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Page metadata coverage:", len(page_meta), "/", page_count)
print("✅ Critical page→module regression checks passed.")


In [ ]:
# Cell 13 — Content-role classifier

def classify_content_role(
    item_type: str,
    text: str,
    heading_path: list[str],
    label: str = "",
) -> str:
    low = (text + " " + " ".join(heading_path)).lower()

    if item_type == "TableItem":
        return "table"
    if item_type == "PictureItem":
        return "figure"

    if any(k in low for k in [
        "pre- and post-tests",
        "pre- and post-tests",
        "pre and post test",
        "pre-test",
        "post-test",
    ]):
        return "pre_post_test"

    if re.search(r"\bcase\s+\d+\b", low):
        if re.search(r"\banswer\b", low):
            return "answer_key"
        return "case_scenario"

    if any(k in low for k in [
        "additional reading", "references", "bibliography"
    ]):
        return "reference"

    if re.search(r"\bactivity\s+\d+\b", low) or re.search(r"\bstep\s+\d+\b", low):
        return "training_activity"

    if any(k in low for k in [
        "who recommends", "who suggests", "recommendation",
        "recommended", "should be referred", "should be given",
        "protocol", "diagnosing high blood pressure",
    ]):
        return "normative_recommendation"

    if any(k in low for k in [
        "implementation remarks", "implementation", "follow-up",
        "referral", "monitoring"
    ]):
        return "implementation_remark"

    if "learning outcome" in low or "competency" in low:
        return "training_objective"

    return "background"

In [ ]:
# Cell 14 — Deterministic clinical metadata extraction

CONDITION_PATTERNS = {
    "hypertension": r"\bhypertension\b|\bhigh blood pressure\b",
    "diabetes": r"\bdiabetes\b|\bdiabetes mellitus\b",
    "cardiovascular_disease": r"\bcardiovascular\b|\bCVD\b",
    "stroke": r"\bstroke\b|\bTIA\b|transient ischaemic",
    "copd": r"\bCOPD\b|chronic obstructive pulmonary",
    "asthma": r"\basthma\b",
    "tobacco": r"\btobacco\b|\bsmok",
    "alcohol": r"\balcohol\b|\bAUDIT\b",
    "obesity": r"\bobesity\b|\boverweight\b|\bBMI\b",
    "diet": r"\bdiet\b|\bsalt\b|\bsugar\b|\bfat\b",
    "physical_activity": r"physical activity|\bexercise\b",
    "oral_cancer": r"oral cancer",
    "breast_cancer": r"breast cancer",
    "cervical_cancer": r"cervical cancer",
    "mental_health": r"depression|mental disorder|anxiety",
    "palliative_care": r"palliative care",
}

INSTRUMENT_PATTERNS = {
    "AUDIT": r"\bAUDIT\b",
    "WHO_ISH_risk_chart": r"WHO/ISH|risk prediction chart",
    "BMI": r"\bBMI\b|body mass index",
    "WHR": r"\bWHR\b|waist.?hip ratio",
    "glucometer": r"\bglucometer\b",
    "peak_flow_meter": r"peak.?flow",
}

DRUG_PATTERNS = {
    "amlodipine": r"\bamlodipine\b",
    "enalapril": r"\benalapril\b",
    "losartan": r"\blosartan\b",
    "telmisartan": r"\btelmi?sartan\b",
    "labetalol": r"\blabetalol\b",
    "hydrochlorothiazide": r"\bhydrochloro?thiazide\b",
    "chlorthalidone": r"\bchlorthalidone\b",
    "metformin": r"\bmetformin\b",
    "gliclazide": r"\bgliclazide\b",
    "aspirin": r"\baspirin\b",
    "atorvastatin": r"\batorvastatin\b",
}

def extract_clinical_metadata(text: str) -> dict:
    conditions = [
        key for key, pat in CONDITION_PATTERNS.items()
        if re.search(pat, text, re.I)
    ]
    instruments = [
        key for key, pat in INSTRUMENT_PATTERNS.items()
        if re.search(pat, text, re.I)
    ]
    drugs = [
        key for key, pat in DRUG_PATTERNS.items()
        if re.search(pat, text, re.I)
    ]

    measures = sorted(set(
        re.findall(
            r"\b(?:\d+(?:\.\d+)?(?:\s*[–-]\s*\d+(?:\.\d+)?)?)\s*"
            r"(?:mmHg|mg/dL|mmol/L|mg|µg|mcg|mL|ml|cm|kg|%)\b",
            text,
            re.I,
        )
    ))

    return {
        "conditions": conditions,
        "instruments": instruments,
        "drug_entities": drugs,
        "measures": measures,
    }

In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 15 — Build a canonical element ledger with exact provenance

heading_stack = {}
element_rows = []

def clean_repeated_noise(text: str, label: str) -> str:
    # Keep headings even if repeated; remove only repeated short lines from body text.
    if "section" in label or "title" in label:
        return text

    kept = []
    for line in text.splitlines():
        norm = normalize_line_for_frequency(line)
        if norm in repeated_candidates and len(norm) <= 120:
            continue
        kept.append(line)
    return "\n".join(kept).strip()

for item, level in tqdm(
    dl_doc.iterate_items(traverse_pictures=True),
    desc="Building element ledger"
):
    item_type = type(item).__name__
    label = label_name(item)
    content_layer = content_layer_name(item)

    # Explicitly exclude furniture/header/footer layers.
    if any(x in label for x in ["page_header", "page_footer"]):
        continue
    if "furniture" in content_layer:
        continue

    prov_list = getattr(item, "prov", None) or []
    prov = prov_list[0] if prov_list else None
    page_no = getattr(prov, "page_no", None)

    if page_no is None or page_no not in page_meta:
        continue

    if isinstance(item, TextItem):
        raw_text = getattr(item, "text", "") or ""
    elif isinstance(item, TableItem):
        try:
            raw_text = item.export_to_markdown(doc=dl_doc)
        except Exception:
            raw_text = ""
    elif isinstance(item, PictureItem):
        try:
            raw_text = item.caption_text(dl_doc) or ""
        except Exception:
            raw_text = ""
    else:
        raw_text = getattr(item, "text", "") or ""

    raw_text = raw_text.strip()
    if not raw_text and item_type not in {"PictureItem"}:
        continue

    # Maintain a simple heading path keyed by hierarchy level.
    is_heading = (
        "section_header" in label
        or label == "title"
        or label.endswith("title")
    )

    if is_heading and raw_text:
        heading_stack[level] = normalize_medical_text(raw_text)
        for deeper in [k for k in list(heading_stack) if k > level]:
            heading_stack.pop(deeper, None)

    heading_path = [
        heading_stack[k]
        for k in sorted(heading_stack)
        if k <= level
    ]

    normalized_text = normalize_medical_text(raw_text)
    normalized_text = clean_repeated_noise(normalized_text, label)

    pmeta = page_meta[page_no]
    role = classify_content_role(
        item_type=item_type,
        text=normalized_text,
        heading_path=heading_path,
        label=label,
    )
    clinical = extract_clinical_metadata(normalized_text)

    doc_ref = getattr(item, "self_ref", "")
    element_id = hashlib.sha1(
        f"{CFG.document_id}|{doc_ref}|{page_no}|{raw_text[:200]}".encode("utf-8")
    ).hexdigest()[:20]

    element_rows.append({
        "element_id": element_id,
        "document_id": CFG.document_id,
        "document_title": CFG.document_title,
        "theme_id": pmeta["theme_id"],
        "theme_title": pmeta["theme_title"],
        "module_id": pmeta["module_id"],
        "module_title": pmeta["module_title"],
        "page_number": page_no,
        "doc_item_ref": doc_ref,
        "item_type": item_type,
        "label": label,
        "content_layer": content_layer,
        "hierarchy_level": int(level),
        "heading_path": heading_path,
        "section_title": heading_path[-1] if heading_path else "",
        "content_role": role,
        "raw_text": raw_text,
        "normalized_text": normalized_text,
        "bbox": safe_bbox(prov),
        "conditions": clinical["conditions"],
        "instruments": clinical["instruments"],
        "drug_entities": clinical["drug_entities"],
        "measures": clinical["measures"],
    })

elements_df = pd.DataFrame(element_rows)
elements_df.to_parquet(ELEMENTS_PATH, index=False)

print("Canonical elements:", len(elements_df))
display(elements_df[[
    "page_number", "module_id", "section_title",
    "item_type", "content_role", "normalized_text"
]].head(20))
print(f'\n⏱ 15 Canonical element ledger: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 16 — Lossless canonical table ledger with duplicate-header protection


def get_table_page(table):
    prov = getattr(table, "prov", None) or []
    return int(prov[0].page_no) if prov else None


def make_unique_headers(original_headers):
    # Internal keys are position-based so duplicate/blank source headers cannot drop columns.
    return [f"col_{i}" for i in range(len(original_headers))]


table_records = []
duplicate_header_tables = 0
failed_table_exports = 0
bad_width_tables = 0

for i, table in enumerate(tqdm(dl_doc.tables, desc="Exporting tables")):
    page_no = get_table_page(table)
    if page_no is None or page_no not in page_meta:
        continue

    try:
        df = table.export_to_dataframe(doc=dl_doc)
    except Exception:
        failed_table_exports += 1
        df = pd.DataFrame()

    original_headers = [normalize_medical_text(str(c)) for c in df.columns]
    safe_columns = make_unique_headers(original_headers)
    safe_df = df.fillna("").astype(str).copy()
    if len(safe_columns):
        safe_df.columns = safe_columns

    if len(original_headers) != len(set(original_headers)):
        duplicate_header_tables += 1

    safe_rows = safe_df.to_dict("records") if not safe_df.empty else []
    if any(len(row) != len(safe_columns) for row in safe_rows):
        bad_width_tables += 1

    try:
        markdown = table.export_to_markdown(doc=dl_doc)
    except Exception:
        markdown = df.to_markdown(index=False) if not df.empty else ""

    try:
        html = table.export_to_html(doc=dl_doc)
    except Exception:
        html = df.to_html(index=False) if not df.empty else ""

    try:
        caption = table.caption_text(dl_doc) or ""
    except Exception:
        caption = ""

    prov = (getattr(table, "prov", None) or [None])[0]
    pmeta = page_meta[page_no]
    table_id = hashlib.sha1(
        f"{CFG.document_id}|table|{i}|{page_no}|{caption}".encode("utf-8")
    ).hexdigest()[:20]

    column_schema = [
        {"key": key, "source_header": original_headers[j], "position": j}
        for j, key in enumerate(safe_columns)
    ]

    table_records.append({
        "table_id": table_id,
        "page_number": page_no,
        "theme_id": pmeta["theme_id"],
        "theme_title": pmeta["theme_title"],
        "module_id": pmeta["module_id"],
        "module_title": pmeta["module_title"],
        "caption": normalize_medical_text(caption),
        "columns": safe_columns,
        "original_headers": original_headers,
        "column_schema": column_schema,
        "rows": safe_rows,
        "markdown": normalize_medical_text(markdown),
        "html": html,
        "bbox": safe_bbox(prov),
        "doc_item_ref": getattr(table, "self_ref", ""),
        "qa": {
            "has_duplicate_headers": len(original_headers) != len(set(original_headers)),
            "row_count": len(safe_rows),
            "column_count": len(safe_columns),
        },
    })

with TABLES_PATH.open("w", encoding="utf-8") as f:
    for rec in table_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Docling tables detected:", len(dl_doc.tables))
print("Tables exported:", len(table_records))
print("Tables with duplicate headers:", duplicate_header_tables)
print("Failed/skipped table exports:", failed_table_exports)
print("Rows with lost/missing columns:", bad_width_tables)
assert bad_width_tables == 0
print(f'\n⏱ 16 Table extraction: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
# Cell 17 — Detect likely multi-page table continuations


def table_header_signature(rec: dict) -> str:
    headers = rec.get("original_headers") or rec.get("columns") or []
    cols = [re.sub(r"\s+", " ", str(c).strip().lower()) for c in headers]
    return "||".join(cols)


table_records_sorted = sorted(table_records, key=lambda x: (x["page_number"], x["table_id"]))
last_by_sig = {}
table_group_map = {}

for rec in table_records_sorted:
    sig = table_header_signature(rec)
    prev = last_by_sig.get(sig)

    if (
        prev
        and rec["page_number"] - prev["page_number"] <= 1
        and rec["module_id"] == prev["module_id"]
        and sig
    ):
        group_id = table_group_map[prev["table_id"]]
    else:
        group_id = "tg_" + hashlib.sha1(
            f"{rec['module_id']}|{rec['page_number']}|{sig}".encode("utf-8")
        ).hexdigest()[:16]

    table_group_map[rec["table_id"]] = group_id
    last_by_sig[sig] = rec

for rec in table_records:
    rec["table_group_id"] = table_group_map[rec["table_id"]]

# Persist the group IDs, too.
with TABLES_PATH.open("w", encoding="utf-8") as f:
    for rec in table_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Table groups:", len(set(table_group_map.values())))


In [ ]:
# Cell 18 — Create a visual QA queue for figure/chart/algorithm-heavy pages

visual_keywords = re.compile(
    r"\b(figure|chart|algorithm|flowchart|risk prediction|pyramid|diagram)\b",
    re.I
)

visual_pages = set(
    audit_df.loc[audit_df["images"] > 0, "page"].tolist()
)

for page_no, text in raw_page_texts.items():
    if visual_keywords.search(text):
        visual_pages.add(page_no)

visual_queue = pd.DataFrame({
    "page": sorted(visual_pages)
})

visual_queue["module_id"] = visual_queue["page"].map(
    lambda p: page_meta[p]["module_id"]
)
visual_queue["reason"] = visual_queue["page"].map(
    lambda p: "keyword_or_image"
)

visual_queue.to_csv(
    CFG.root / "visual_qa" / "visual_pages.csv",
    index=False
)

print("Visual QA pages:", len(visual_queue))
display(visual_queue.head(30))

In [ ]:
# Cell 19 — Optional: render selected source pages for human visual QA

def render_pdf_pages(page_numbers, dpi=160):
    out_dir = CFG.root / "visual_qa" / "rendered_pages"
    out_dir.mkdir(parents=True, exist_ok=True)

    rendered = []
    scale = dpi / 72.0
    matrix = fitz.Matrix(scale, scale)

    for p in page_numbers:
        out = out_dir / f"page_{p:04d}.png"
        if not out.exists():
            pix = pdf[p - 1].get_pixmap(matrix=matrix, alpha=False)
            pix.save(out)
        rendered.append(out)

    return rendered

# Example:
# rendered = render_pdf_pages([86, 87, 126, 127, 129, 218])
# rendered

In [ ]:
# Cell 20 — Load tokenizer used for exact token-aware hierarchical chunking

from transformers import AutoTokenizer

chunk_tokenizer = AutoTokenizer.from_pretrained(
    CFG.embedding_model,
    use_fast=True,
)

def token_count(text: str) -> int:
    return len(
        chunk_tokenizer.encode(
            text,
            add_special_tokens=False,
        )
    )

def split_token_windows(
    text: str,
    max_tokens: int,
    overlap_ratio: float,
) -> list[str]:
    ids = chunk_tokenizer.encode(
        text,
        add_special_tokens=False,
    )
    if len(ids) <= max_tokens:
        return [text.strip()] if text.strip() else []

    overlap = max(1, int(max_tokens * overlap_ratio))
    step = max_tokens - overlap

    windows = []
    for start in range(0, len(ids), step):
        piece_ids = ids[start:start + max_tokens]
        if not piece_ids:
            break
        piece = chunk_tokenizer.decode(
            piece_ids,
            skip_special_tokens=True,
        ).strip()
        if piece:
            windows.append(piece)
        if start + max_tokens >= len(ids):
            break
    return windows

print("Tokenizer:", CFG.embedding_model)

In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 21 — Build L3 text parents, robust to Parquet list/ndarray metadata

import ast


def metadata_to_list(value):
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return [x for item in value.tolist() for x in metadata_to_list(item)]
    if isinstance(value, (list, tuple, set, pd.Series)):
        return [x for item in list(value) for x in metadata_to_list(item)]
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        if s.startswith("[") and s.endswith("]"):
            try:
                return metadata_to_list(ast.literal_eval(s))
            except Exception:
                pass
        return [s]
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    return [str(value)]


def unique_clean(values):
    out = []
    seen = set()
    for v in metadata_to_list(values):
        s = str(v).strip()
        if not s or s.lower() in {"nan", "none", "null"}:
            continue
        if s not in seen:
            seen.add(s)
            out.append(s)
    return sorted(out)


def stable_id(prefix: str, *parts) -> str:
    payload = "|".join(str(x) for x in parts)
    return f"{prefix}_" + hashlib.sha1(payload.encode("utf-8")).hexdigest()[:20]


def overlap_tail(units, max_overlap_tokens):
    selected, total = [], 0
    for u in reversed(units):
        txt = str(u.get("normalized_text", "") or "")
        c = token_count(txt)
        if total + c > max_overlap_tokens and selected:
            break
        selected.append(u)
        total += c
        if total >= max_overlap_tokens:
            break
    return list(reversed(selected))


text_elements = elements_df[~elements_df["item_type"].isin(["TableItem"])].copy()
text_elements["normalized_text"] = text_elements["normalized_text"].fillna("").astype(str)
text_elements = text_elements[text_elements["normalized_text"].str.strip().str.len() > 0].copy()

parent_rows = []
group_cols = ["theme_id", "module_id", "section_title"]

for group_key, group in tqdm(
    text_elements.sort_values(["page_number", "hierarchy_level"]).groupby(
        group_cols, dropna=False, sort=False
    ),
    desc="Building text parents",
):
    group = group.to_dict("records")
    buffer_units, buffer_tokens = [], 0
    parent_seq_counter = [0]

    def flush_parent(units):
        if not units:
            return []
        joined = "\n\n".join(
            str(x.get("normalized_text", "")).strip()
            for x in units if str(x.get("normalized_text", "")).strip()
        ).strip()
        if not joined:
            return []

        pieces = split_token_windows(joined, CFG.parent_max_tokens, CFG.parent_overlap_ratio)
        emitted = []
        for sub_seq, piece in enumerate(pieces):
            pages = sorted(set(int(x["page_number"]) for x in units))
            refs = list(dict.fromkeys(x.get("doc_item_ref", "") for x in units if x.get("doc_item_ref", "")))
            roles = list(dict.fromkeys(str(x.get("content_role", "")) for x in units if str(x.get("content_role", ""))))
            conditions = unique_clean([x.get("conditions", []) for x in units])
            instruments = unique_clean([x.get("instruments", []) for x in units])
            drugs = unique_clean([x.get("drug_entities", []) for x in units])

            first = units[0]
            parent_id = stable_id(
                "parent", first["module_id"], first.get("section_title", ""),
                pages[0], parent_seq_counter[0], sub_seq, piece[:150]
            )
            emitted.append({
                "parent_id": parent_id,
                "node_level": "L3_parent",
                "document_id": CFG.document_id,
                "theme_id": first["theme_id"],
                "theme_title": first["theme_title"],
                "module_id": first["module_id"],
                "module_title": first["module_title"],
                "section_title": first.get("section_title", ""),
                "content_role": roles[0] if len(roles) == 1 else "mixed",
                "content_roles": roles,
                "page_start": min(pages),
                "page_end": max(pages),
                "pages": pages,
                "doc_item_refs": refs,
                "conditions": conditions,
                "instruments": instruments,
                "drug_entities": drugs,
                "table_id": None,
                "table_group_id": None,
                "text": piece,
                "token_count": token_count(piece),
            })
            parent_seq_counter[0] += 1
        return emitted

    for unit in group:
        unit_tokens = token_count(str(unit.get("normalized_text", "")))
        if unit_tokens > CFG.parent_max_tokens:
            parent_rows.extend(flush_parent(buffer_units))
            buffer_units, buffer_tokens = [], 0
            parent_rows.extend(flush_parent([unit]))
            continue

        if buffer_units and buffer_tokens + unit_tokens > CFG.parent_max_tokens:
            parent_rows.extend(flush_parent(buffer_units))
            overlap_budget = int(CFG.parent_max_tokens * CFG.parent_overlap_ratio)
            buffer_units = overlap_tail(buffer_units, overlap_budget)
            buffer_tokens = sum(token_count(str(x.get("normalized_text", ""))) for x in buffer_units)

        buffer_units.append(unit)
        buffer_tokens += unit_tokens

    parent_rows.extend(flush_parent(buffer_units))

print("Text parents:", len(parent_rows))
print(f'\n⏱ 21 L3 text parent chunking: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 22 — Dedicated semantic table parents (AUDIT/numeric/table friendly)


def clean_header(h, position):
    s = normalize_medical_text(str(h or "")).strip()
    return s if s else f"Column {position + 1}"


def row_to_semantic_line(rec: dict, row: dict) -> str:
    safe_cols = rec.get("columns", [])
    headers = rec.get("original_headers", [])
    parts = []
    for i, key in enumerate(safe_cols):
        value = normalize_medical_text(str(row.get(key, ""))).strip()
        if not value:
            continue
        label = clean_header(headers[i] if i < len(headers) else "", i)
        # Natural key:value representation is easier for embeddings/rerankers than
        # anonymous col_0/col_1 markdown alone.
        parts.append(f"{label}: {value}")
    return " | ".join(parts)


def build_table_parent_texts(rec: dict) -> list[str]:
    rows = rec.get("rows", [])
    caption = normalize_medical_text(rec.get("caption", "")).strip()
    headers = rec.get("original_headers", [])

    prefix_lines = [
        f"Module: {rec.get('module_id', '')} — {rec.get('module_title', '')}",
        f"Table caption: {caption or 'Table'}",
    ]
    if headers:
        display_headers = [clean_header(h, i) for i, h in enumerate(headers)]
        prefix_lines.append("Table columns: " + " | ".join(display_headers))

    prefix = "\n".join(prefix_lines).strip()
    semantic_rows = [row_to_semantic_line(rec, row) for row in rows]
    semantic_rows = [x for x in semantic_rows if x]

    # Fallback to Docling markdown if dataframe rows are unavailable.
    if not semantic_rows:
        fallback = normalize_medical_text(rec.get("markdown", ""))
        text = (prefix + "\n\n" + fallback).strip()
        return split_token_windows(text, CFG.parent_max_tokens, CFG.parent_overlap_ratio)

    max_tokens = CFG.parent_max_tokens
    overlap_budget = int(max_tokens * CFG.parent_overlap_ratio)
    outputs, current = [], []

    def make_text(lines):
        return (prefix + "\n\nRows:\n" + "\n".join(lines)).strip()

    for line in semantic_rows:
        candidate = make_text(current + [line])
        if current and token_count(candidate) > max_tokens:
            outputs.append(make_text(current))
            tail, tail_tokens = [], 0
            for old in reversed(current):
                t = token_count(old)
                if tail and tail_tokens + t > overlap_budget:
                    break
                tail.append(old)
                tail_tokens += t
            current = list(reversed(tail))
        current.append(line)

    if current:
        outputs.append(make_text(current))
    return outputs


for rec in tqdm(table_records, desc="Building semantic table parents"):
    pieces = build_table_parent_texts(rec)
    pmeta = page_meta[int(rec["page_number"])]

    for seq, piece in enumerate(pieces):
        clinical = extract_clinical_metadata(piece)
        parent_id = stable_id("parent_table", rec["table_id"], seq, piece[:160])
        parent_rows.append({
            "parent_id": parent_id,
            "node_level": "L3_parent",
            "document_id": CFG.document_id,
            "theme_id": pmeta["theme_id"],
            "theme_title": pmeta["theme_title"],
            "module_id": pmeta["module_id"],
            "module_title": pmeta["module_title"],
            "section_title": rec.get("caption", "") or "Table",
            "content_role": "table",
            "content_roles": ["table"],
            "page_start": int(rec["page_number"]),
            "page_end": int(rec["page_number"]),
            "pages": [int(rec["page_number"])],
            "doc_item_refs": [rec.get("doc_item_ref", "")],
            "conditions": clinical["conditions"],
            "instruments": clinical["instruments"],
            "drug_entities": clinical["drug_entities"],
            "table_id": rec["table_id"],
            "table_group_id": rec.get("table_group_id"),
            "text": piece,
            "token_count": token_count(piece),
        })

parents_df = pd.DataFrame(parent_rows)
parents_df.to_parquet(PARENTS_PATH, index=False)

print("Total L3 parents:", len(parents_df))
print("Table parents:", int((parents_df["content_role"] == "table").sum()))
display(parents_df["token_count"].describe())
print(f'\n⏱ 22 L3 table chunking: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 23 — Build L4 leaves (100–250-token class), embedding leaves only

leaf_rows = []

for parent in tqdm(
    parents_df.to_dict("records"),
    desc="Building L4 leaves"
):
    pieces = split_token_windows(
        parent["text"],
        CFG.leaf_max_tokens,
        CFG.leaf_overlap_ratio,
    )

    for seq, piece in enumerate(pieces):
        leaf_id = stable_id(
            "leaf",
            parent["parent_id"],
            seq,
            piece[:160],
        )

        # Contextualized retrieval text; raw leaf remains separately available.
        context_prefix = (
            f"Theme: {parent['theme_title']}\n"
            f"Module: {parent['module_id']} — {parent['module_title']}\n"
            f"Section: {parent['section_title']}\n"
            f"Content role: {parent['content_role']}\n"
        )

        embedding_text = context_prefix + "\n" + piece

        leaf_rows.append({
            "leaf_id": leaf_id,
            "parent_id": parent["parent_id"],
            "node_level": "L4_leaf",
            "document_id": parent["document_id"],
            "theme_id": parent["theme_id"],
            "theme_title": parent["theme_title"],
            "module_id": parent["module_id"],
            "module_title": parent["module_title"],
            "section_title": parent["section_title"],
            "content_role": parent["content_role"],
            "page_start": parent["page_start"],
            "page_end": parent["page_end"],
            "pages": parent["pages"],
            "conditions": parent["conditions"],
            "instruments": parent["instruments"],
            "drug_entities": parent["drug_entities"],
            "table_id": parent["table_id"],
            "table_group_id": parent["table_group_id"],
            "text": piece,
            "embedding_text": embedding_text,
            "token_count": token_count(piece),
        })

leaves_df = pd.DataFrame(leaf_rows)
leaves_df.to_parquet(LEAVES_PATH, index=False)

print("Total L4 leaves:", len(leaves_df))
display(leaves_df["token_count"].describe())
print(f'\n⏱ 23 L4 leaf chunking: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
# Cell 24 — Hierarchy integrity checks (resume-safe)

if "page_count" not in globals():
    if MANIFEST_PATH.exists():
        _manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        page_count = int(_manifest.get("page_count", CFG.expected_pages))
    else:
        page_count = int(CFG.expected_pages)

parent_ids = set(parents_df["parent_id"])
leaf_parent_ids = set(leaves_df["parent_id"])

assert leaf_parent_ids.issubset(parent_ids)
assert leaves_df["leaf_id"].is_unique
assert parents_df["parent_id"].is_unique
assert leaves_df["page_start"].between(1, page_count).all()
assert leaves_df["page_end"].between(1, page_count).all()
assert parents_df["page_start"].between(1, page_count).all()
assert parents_df["page_end"].between(1, page_count).all()

oversized_parents = parents_df[parents_df["token_count"] > CFG.parent_max_tokens + 20]
oversized_leaves = leaves_df[leaves_df["token_count"] > CFG.leaf_max_tokens + 20]

assert len(oversized_parents) == 0
assert len(oversized_leaves) == 0

print("Parents:", len(parents_df))
print("Leaves:", len(leaves_df))
print("Oversized parents:", len(oversized_parents))
print("Oversized leaves:", len(oversized_leaves))
print("Leaf/parent ratio:", round(len(leaves_df) / max(len(parents_df), 1), 2))
print("✅ Hierarchy integrity checks passed.")


In [ ]:
# Cell 24B — Release ingestion memory before loading BGE-M3
import gc, os, psutil

for name in [
    "dl_doc", "converter", "conv_result", "pipeline_options", "pdf",
    "element_rows", "page_audit", "audit_df", "raw_page_texts",
    "text_elements", "parent_rows", "module_hits", "hit_df",
]:
    if name in globals():
        del globals()[name]

for _ in range(3):
    gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

process = psutil.Process(os.getpid())
print("Python RAM GB:", round(process.memory_info().rss / 1024**3, 2))
print("Available RAM GB:", round(psutil.virtual_memory().available / 1024**3, 2))
print("✅ Ready to load BGE-M3.")


In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 25 — Load multilingual BGE-M3 embedding model

from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    CFG.embedding_model,
    device=DEVICE,
)

print("Embedding model loaded:", CFG.embedding_model)
print(f'\n⏱ 25 Load embedding model: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
# Cell 25B — Automatic embedding batch-size benchmark
# Finds the fastest SAFE batch size on this GPU before the full corpus embedding run.

def benchmark_embedding_batch_sizes(
    model,
    sample_texts,
    candidates=None,
    max_peak_pct=90.0,
):
    if candidates is None:
        candidates = [8, 16, 24, 32, 48, 64]

    # Repeat the sample enough to exercise batching consistently.
    sample_texts = list(sample_texts)
    if not sample_texts:
        raise ValueError("No sample texts available for batch benchmark.")

    while len(sample_texts) < 128:
        sample_texts = sample_texts + sample_texts
    sample_texts = sample_texts[:128]

    results = []

    for bs in candidates:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        start = time.perf_counter()

        try:
            _ = model.encode(
                sample_texts,
                batch_size=bs,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=True,
            )

            elapsed = time.perf_counter() - start
            throughput = len(sample_texts) / max(elapsed, 1e-9)

            if torch.cuda.is_available():
                total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
                peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
                peak_pct = peak / total * 100.0
            else:
                peak = 0.0
                peak_pct = 0.0

            results.append({
                "batch_size": bs,
                "status": "OK",
                "seconds": elapsed,
                "texts_per_second": throughput,
                "peak_vram_gb": peak,
                "peak_vram_pct": peak_pct,
            })

            print(
                f"Embedding BS={bs:>2} | "
                f"{throughput:>7.1f} texts/s | "
                f"Peak VRAM={peak:>5.2f} GB ({peak_pct:>5.1f}%)"
            )

        except torch.cuda.OutOfMemoryError:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            results.append({
                "batch_size": bs,
                "status": "OOM",
                "seconds": None,
                "texts_per_second": 0.0,
                "peak_vram_gb": None,
                "peak_vram_pct": None,
            })

            print(f"Embedding BS={bs:>2} | OOM")

    df = pd.DataFrame(results)

    safe = df[
        (df["status"] == "OK")
        & (df["peak_vram_pct"].fillna(0) <= max_peak_pct)
    ].copy()

    if safe.empty:
        chosen = int(
            df.loc[df["status"] == "OK", "batch_size"].min()
        )
    else:
        # Pick the fastest safe candidate, not blindly the largest.
        chosen = int(
            safe.sort_values(
                ["texts_per_second", "batch_size"],
                ascending=[False, False],
            ).iloc[0]["batch_size"]
        )

    return df, chosen

embedding_batch_benchmark_df, chosen_embedding_bs = benchmark_embedding_batch_sizes(
    model=embedder,
    sample_texts=leaves_df["embedding_text"].head(128).tolist(),
)

CFG.embedding_batch_size = chosen_embedding_bs

print("\n✅ Selected embedding batch size:", CFG.embedding_batch_size)
display(embedding_batch_benchmark_df)

In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 26 — Fingerprinted L4 embeddings with OOM-safe dynamic batching


def leaves_fingerprint(df: pd.DataFrame) -> str:
    h = hashlib.sha256()
    for leaf_id, text in zip(df["leaf_id"].astype(str), df["embedding_text"].astype(str)):
        h.update(leaf_id.encode("utf-8")); h.update(b"\0")
        h.update(text.encode("utf-8")); h.update(b"\n")
    return h.hexdigest()


def encode_with_oom_backoff(model, texts, batch_size, min_batch_size=1):
    current_bs = int(batch_size)
    while current_bs >= min_batch_size:
        try:
            print(f"Embedding with batch_size={current_bs}")
            embeddings = model.encode(
                texts,
                batch_size=current_bs,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
            ).astype(np.float32)
            return embeddings, current_bs
        except torch.cuda.OutOfMemoryError:
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            if current_bs == min_batch_size:
                raise
            current_bs = max(min_batch_size, current_bs // 2)
            print(f"CUDA OOM. Retrying with batch_size={current_bs}")


LEAF_FINGERPRINT = leaves_fingerprint(leaves_df)
cache_ok = False
if EMBEDDINGS_PATH.exists() and EMBEDDINGS_META_PATH.exists():
    try:
        meta = json.loads(EMBEDDINGS_META_PATH.read_text(encoding="utf-8"))
        cache_ok = (
            meta.get("leaf_fingerprint") == LEAF_FINGERPRINT
            and meta.get("embedding_model") == CFG.embedding_model
        )
    except Exception:
        cache_ok = False

if cache_ok:
    leaf_embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
    if len(leaf_embeddings) != len(leaves_df):
        cache_ok = False

if not cache_ok:
    texts = leaves_df["embedding_text"].tolist()
    leaf_embeddings, used_embedding_bs = encode_with_oom_backoff(
        embedder, texts, CFG.embedding_batch_size
    )
    CFG.embedding_batch_size = used_embedding_bs
    np.save(EMBEDDINGS_PATH, leaf_embeddings)
    EMBEDDINGS_META_PATH.write_text(json.dumps({
        "leaf_fingerprint": LEAF_FINGERPRINT,
        "embedding_model": CFG.embedding_model,
        "shape": list(leaf_embeddings.shape),
    }, indent=2), encoding="utf-8")
    print("Saved embeddings:", leaf_embeddings.shape)
else:
    print("Loaded fingerprint-matched cached embeddings:", leaf_embeddings.shape)

print(f'\n⏱ 26 Full-corpus embedding: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 27 — Fingerprinted, local-lock-safe persistent Qdrant index

from qdrant_client import QdrantClient, models
import shutil, gc, uuid

COLLECTION = "who_pen_leaves"


def close_global_qdrant():
    global qdrant
    if "qdrant" in globals():
        try:
            qdrant.close()
        except Exception:
            pass
        try:
            del qdrant
        except Exception:
            pass
        gc.collect()


def qdrant_meta_matches():
    if not QDRANT_META_PATH.exists():
        return False
    try:
        meta = json.loads(QDRANT_META_PATH.read_text(encoding="utf-8"))
        return (
            meta.get("leaf_fingerprint") == LEAF_FINGERPRINT
            and meta.get("embedding_model") == CFG.embedding_model
            and int(meta.get("points", -1)) == len(leaves_df)
        )
    except Exception:
        return False


# Never double-open the same local storage.
close_global_qdrant()

if QDRANT_DIR.exists() and not qdrant_meta_matches():
    shutil.rmtree(QDRANT_DIR)

QDRANT_DIR.mkdir(parents=True, exist_ok=True)
qdrant = QdrantClient(path=str(QDRANT_DIR))

existing_count = 0
if qdrant.collection_exists(COLLECTION):
    existing_count = qdrant.get_collection(COLLECTION).points_count or 0

needs_rebuild = (not qdrant_meta_matches()) or existing_count != len(leaves_df)

if needs_rebuild:
    if qdrant.collection_exists(COLLECTION):
        qdrant.delete_collection(COLLECTION)

    vector_size = int(leaf_embeddings.shape[1])
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE),
    )

    points = []
    for i, row in enumerate(tqdm(leaves_df.to_dict("records"), desc="Writing Qdrant")):
        qid = str(uuid.uuid5(uuid.NAMESPACE_URL, f"{CFG.document_id}:{row['leaf_id']}"))
        payload = {
            "leaf_id": row["leaf_id"],
            "parent_id": row["parent_id"],
            "theme_id": row["theme_id"],
            "module_id": row["module_id"],
            "section_title": row["section_title"],
            "content_role": row["content_role"],
            "page_start": int(row["page_start"]),
            "page_end": int(row["page_end"]),
        }
        points.append(models.PointStruct(
            id=qid,
            vector=np.asarray(leaf_embeddings[i], dtype=np.float32).tolist(),
            payload=payload,
        ))
        if len(points) >= 128:
            qdrant.upsert(collection_name=COLLECTION, points=points, wait=True)
            points = []
    if points:
        qdrant.upsert(collection_name=COLLECTION, points=points, wait=True)

    QDRANT_META_PATH.write_text(json.dumps({
        "leaf_fingerprint": LEAF_FINGERPRINT,
        "embedding_model": CFG.embedding_model,
        "points": len(leaves_df),
    }, indent=2), encoding="utf-8")

final_count = qdrant.get_collection(COLLECTION).points_count or 0
assert final_count == len(leaves_df)
print("Qdrant points:", final_count)
print("✅ Qdrant index fingerprint matches current leaves.")
print(f'\n⏱ 27 Qdrant indexing: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 28 — Fingerprinted BM25 sparse index

from rank_bm25 import BM25Okapi


def lexical_tokenize(text: str) -> list[str]:
    text = text.lower()
    return re.findall(
        r"[a-z]+(?:-[a-z]+)?|\d+(?:\.\d+)?|mmhg|mg/dl|mmol/l|µg|mcg|mg|%|[><=≤≥]+",
        text,
        flags=re.I,
    )

bm25_bundle = None
if BM25_PATH.exists():
    try:
        candidate = joblib.load(BM25_PATH)
        if (
            candidate.get("leaf_fingerprint") == LEAF_FINGERPRINT
            and candidate.get("leaf_ids") == leaves_df["leaf_id"].tolist()
        ):
            bm25_bundle = candidate
    except Exception:
        bm25_bundle = None

if bm25_bundle is None:
    tokenized_corpus = [
        lexical_tokenize(t)
        for t in tqdm(leaves_df["embedding_text"].tolist(), desc="Tokenizing BM25 corpus")
    ]
    bm25 = BM25Okapi(tokenized_corpus)
    bm25_bundle = {
        "bm25": bm25,
        "tokenized_corpus": tokenized_corpus,
        "leaf_ids": leaves_df["leaf_id"].tolist(),
        "leaf_fingerprint": LEAF_FINGERPRINT,
    }
    joblib.dump(bm25_bundle, BM25_PATH)
    print("Saved new fingerprinted BM25 index.")
else:
    bm25 = bm25_bundle["bm25"]
    tokenized_corpus = bm25_bundle["tokenized_corpus"]
    print("Loaded fingerprint-matched BM25 index.")

print("BM25 documents:", len(tokenized_corpus))
print(f'\n⏱ 28 BM25 indexing: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
# Cell 29 — OpenAI-compatible LLM client for query planning, generation and verification

def require_llm_config():
    if not CFG.llm_model:
        raise RuntimeError(
            "Set environment variable LLM_MODEL before running LLM cells."
        )
    if not CFG.llm_api_key:
        raise RuntimeError(
            "Set environment variable LLM_API_KEY before running LLM cells."
        )

def parse_json_object(text: str) -> dict:
    text = text.strip()

    # Remove common markdown fences.
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError("Could not parse JSON from model response.")

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
)
def llm_json(
    system_prompt: str,
    user_prompt: str,
    model: Optional[str] = None,
) -> dict:
    require_llm_config()

    model = model or CFG.llm_model
    url = CFG.llm_base_url.rstrip("/") + "/chat/completions"

    headers = {
        "Authorization": f"Bearer {CFG.llm_api_key}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "response_format": {"type": "json_object"},
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=CFG.request_timeout,
    )

    # Some OpenAI-compatible servers do not support response_format.
    if response.status_code >= 400:
        payload.pop("response_format", None)
        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=CFG.request_timeout,
        )

    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"]
    return parse_json_object(content)

In [ ]:
# Cell 30 — Resume-safe conservative query planner

class QueryPlan(BaseModel):
    semantic_query: str
    lexical_query: str
    theme_id: Optional[str] = None
    module_id: Optional[str] = None
    content_roles: list[str] = Field(default_factory=list)
    filter_confidence: float = 0.0
    patient_specific: bool = False
    out_of_scope: bool = False
    notes: str = ""


def clean_unique_values(series):
    values = set()
    for value in series:
        if value is None:
            continue
        try:
            if pd.isna(value):
                continue
        except Exception:
            pass
        s = str(value).strip()
        if s and s.lower() not in {"nan", "none", "null"}:
            values.add(s)
    return values

# Route only to metadata that actually exists in the indexed L4 leaves.
VALID_THEME_IDS = clean_unique_values(leaves_df["theme_id"])
VALID_MODULE_IDS = clean_unique_values(leaves_df["module_id"])
VALID_CONTENT_ROLES = clean_unique_values(leaves_df["content_role"])


def fallback_query_plan(query: str) -> QueryPlan:
    return QueryPlan(
        semantic_query=query,
        lexical_query=query,
        filter_confidence=0.0,
        notes="LLM planner unavailable; global retrieval fallback.",
    )


def plan_query(query: str, use_llm: bool = True) -> QueryPlan:
    if not use_llm or not CFG.llm_model or not CFG.llm_api_key:
        return fallback_query_plan(query)

    system = '''
You are a conservative query planner for a WHO PEN clinical evidence retrieval system.
Return JSON only.

Rules:
- Preserve the user's clinical meaning.
- semantic_query: concise retrieval query; prefer English clinical terminology even for Arabic input.
- lexical_query: exact clinical terms, acronyms, thresholds, units, drug names and instrument names useful for BM25.
- Routing is OPTIONAL. Never invent a theme/module.
- Hard filters should be proposed only with very high confidence.
- patient_specific=true only for a real/presented patient scenario.
- out_of_scope=true only when WHO PEN clearly does not cover the topic.
- When uncertain, prefer global retrieval over a wrong restriction.
'''

    payload = {
        "query": query,
        "valid_theme_ids": sorted(VALID_THEME_IDS),
        "valid_module_ids": sorted(VALID_MODULE_IDS),
        "valid_content_roles": sorted(VALID_CONTENT_ROLES),
        "required_json_schema": QueryPlan.model_json_schema(),
    }

    try:
        raw = llm_json(system, json.dumps(payload, ensure_ascii=False))
        plan = QueryPlan.model_validate(raw)
    except Exception as e:
        print("Planner fallback:", repr(e))
        return fallback_query_plan(query)

    if plan.theme_id not in VALID_THEME_IDS:
        plan.theme_id = None
    if plan.module_id not in VALID_MODULE_IDS:
        plan.module_id = None
    plan.content_roles = [x for x in plan.content_roles if x in VALID_CONTENT_ROLES]
    plan.filter_confidence = float(max(0.0, min(1.0, plan.filter_confidence)))
    return plan

print("Valid themes:", sorted(VALID_THEME_IDS))
print("Valid modules:", sorted(VALID_MODULE_IDS))
print("Valid content roles:", sorted(VALID_CONTENT_ROLES))
print("✅ Query planner ready.")


In [ ]:
# Cell 31 — Dense, BM25 and RRF retrieval

leaf_lookup = leaves_df.set_index("leaf_id").to_dict("index")
parent_lookup = parents_df.set_index("parent_id").to_dict("index")

def metadata_match(
    row: dict,
    plan: QueryPlan,
    hard_content_role: bool = False,
) -> bool:
    """
    Conservative hard routing.

    theme_id/module_id may be hard filters at high confidence.
    content_role is advisory by default because valid evidence can live in
    background, tables, training activities, or recommendations.
    """
    if plan.theme_id and row["theme_id"] != plan.theme_id:
        return False
    if plan.module_id and row["module_id"] != plan.module_id:
        return False
    if (
        hard_content_role
        and plan.content_roles
        and row["content_role"] not in plan.content_roles
    ):
        return False
    return True

def dense_retrieve(
    semantic_query: str,
    plan: QueryPlan,
    k: int,
    use_hard_filter: bool = True,
) -> list[dict]:
    qvec = embedder.encode(
        [semantic_query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0].astype(np.float32)

    # Retrieve extra candidates, then apply conservative Python-side filters.
    result = qdrant.query_points(
        collection_name=COLLECTION,
        query=qvec.tolist(),
        with_payload=True,
        limit=min(max(k * 3, k), len(leaves_df)),
    ).points

    out = []
    for rank, point in enumerate(result, start=1):
        leaf_id = point.payload["leaf_id"]
        row = leaf_lookup[leaf_id]

        if (
            use_hard_filter
            and plan.filter_confidence >= CFG.hard_filter_confidence
            and not metadata_match(row, plan)
        ):
            continue

        out.append({
            "leaf_id": leaf_id,
            "rank": rank,
            "score": float(point.score),
            "source": "dense",
        })
        if len(out) >= k:
            break

    return out

def bm25_retrieve(
    lexical_query: str,
    plan: QueryPlan,
    k: int,
    use_hard_filter: bool = True,
) -> list[dict]:
    q_tokens = lexical_tokenize(lexical_query)
    scores = bm25.get_scores(q_tokens)
    order = np.argsort(scores)[::-1]

    out = []
    for idx in order:
        leaf_id = leaves_df.iloc[int(idx)]["leaf_id"]
        row = leaf_lookup[leaf_id]

        if (
            use_hard_filter
            and plan.filter_confidence >= CFG.hard_filter_confidence
            and not metadata_match(row, plan)
        ):
            continue

        out.append({
            "leaf_id": leaf_id,
            "rank": len(out) + 1,
            "score": float(scores[int(idx)]),
            "source": "bm25",
        })
        if len(out) >= k:
            break

    return out

def reciprocal_rank_fusion(
    result_lists: list[list[dict]],
    k_constant: int = 60,
) -> list[dict]:
    scores = defaultdict(float)
    ranks = defaultdict(dict)

    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            leaf_id = item["leaf_id"]
            scores[leaf_id] += 1.0 / (k_constant + rank)
            ranks[leaf_id][item["source"]] = rank

    fused = [
        {
            "leaf_id": leaf_id,
            "rrf_score": score,
            "source_ranks": ranks[leaf_id],
        }
        for leaf_id, score in scores.items()
    ]

    fused.sort(key=lambda x: x["rrf_score"], reverse=True)
    return fused

def apply_parent_cap(fused: list[dict]) -> list[dict]:
    per_parent = Counter()
    out = []

    for item in fused:
        parent_id = leaf_lookup[item["leaf_id"]]["parent_id"]
        if per_parent[parent_id] >= CFG.max_leaves_per_parent:
            continue
        per_parent[parent_id] += 1
        out.append(item)
        if len(out) >= CFG.fusion_k:
            break

    return out


In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 32 — Load multilingual BGE cross-encoder reranker

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

rerank_tokenizer = AutoTokenizer.from_pretrained(
    CFG.reranker_model
)

rerank_model = AutoModelForSequenceClassification.from_pretrained(
    CFG.reranker_model,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)

rerank_model = rerank_model.to(DEVICE)
rerank_model.eval()

print("Reranker loaded:", CFG.reranker_model)
print(f'\n⏱ 32 Load reranker model: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
# Cell 32B — Automatic reranker batch-size benchmark
# Reranking is online inference, not training. Candidate pool is capped at CFG.rerank_k.

@torch.inference_mode()
def benchmark_reranker_batch_sizes(
    tokenizer,
    model,
    pairs,
    candidates=None,
    max_peak_pct=90.0,
):
    if candidates is None:
        candidates = [2, 4, 8, 12, 16, 24, 32]

    pairs = list(pairs)
    if not pairs:
        raise ValueError("No reranker pairs available for benchmark.")

    # Build at least 32 pairs to measure batching; actual online rerank_k is 30.
    while len(pairs) < 32:
        pairs = pairs + pairs
    pairs = pairs[:32]

    results = []

    for bs in candidates:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        start = time.perf_counter()

        try:
            n_scored = 0

            for st in range(0, len(pairs), bs):
                batch_pairs = pairs[st:st + bs]

                inputs = tokenizer(
                    batch_pairs,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt",
                ).to(DEVICE)

                logits = model(
                    **inputs,
                    return_dict=True,
                ).logits.view(-1)

                n_scored += int(logits.numel())

            elapsed = time.perf_counter() - start
            throughput = n_scored / max(elapsed, 1e-9)

            if torch.cuda.is_available():
                total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
                peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
                peak_pct = peak / total * 100.0
            else:
                peak = 0.0
                peak_pct = 0.0

            results.append({
                "batch_size": bs,
                "status": "OK",
                "seconds": elapsed,
                "pairs_per_second": throughput,
                "peak_vram_gb": peak,
                "peak_vram_pct": peak_pct,
            })

            print(
                f"Reranker BS={bs:>2} | "
                f"{throughput:>7.1f} pairs/s | "
                f"Peak VRAM={peak:>5.2f} GB ({peak_pct:>5.1f}%)"
            )

        except torch.cuda.OutOfMemoryError:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            results.append({
                "batch_size": bs,
                "status": "OOM",
                "seconds": None,
                "pairs_per_second": 0.0,
                "peak_vram_gb": None,
                "peak_vram_pct": None,
            })

            print(f"Reranker BS={bs:>2} | OOM")

    df = pd.DataFrame(results)

    safe = df[
        (df["status"] == "OK")
        & (df["peak_vram_pct"].fillna(0) <= max_peak_pct)
    ].copy()

    if safe.empty:
        chosen = int(
            df.loc[df["status"] == "OK", "batch_size"].min()
        )
    else:
        chosen = int(
            safe.sort_values(
                ["pairs_per_second", "batch_size"],
                ascending=[False, False],
            ).iloc[0]["batch_size"]
        )

    # No point selecting a batch larger than the online candidate pool.
    chosen = min(chosen, int(CFG.rerank_k))

    return df, chosen

benchmark_query = (
    "What blood pressure criteria are used to diagnose raised blood pressure?"
)

sample_rerank_pairs = [
    [benchmark_query, text]
    for text in leaves_df["embedding_text"].head(32).tolist()
]

reranker_batch_benchmark_df, chosen_reranker_bs = benchmark_reranker_batch_sizes(
    tokenizer=rerank_tokenizer,
    model=rerank_model,
    pairs=sample_rerank_pairs,
)

CFG.reranker_batch_size = chosen_reranker_bs

print("\n✅ Selected reranker batch size:", CFG.reranker_batch_size)
display(reranker_batch_benchmark_df)

In [ ]:
# Cell 33 — Cross-encoder re-ranking with OOM-safe dynamic batching

@torch.inference_mode()
def rerank_candidates(
    query: str,
    candidates: list[dict],
    top_k: int = None,
) -> list[dict]:
    top_k = top_k or CFG.final_leaf_k

    candidates = candidates[:CFG.rerank_k]
    if not candidates:
        return []

    pairs = [
        [query, leaf_lookup[x["leaf_id"]]["embedding_text"]]
        for x in candidates
    ]

    current_bs = int(CFG.reranker_batch_size)

    while current_bs >= 1:
        try:
            all_logits = []

            for start in range(0, len(pairs), current_bs):
                batch_pairs = pairs[start:start + current_bs]

                inputs = rerank_tokenizer(
                    batch_pairs,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt",
                ).to(DEVICE)

                logits = rerank_model(
                    **inputs,
                    return_dict=True,
                ).logits.view(-1).float()

                all_logits.extend(
                    logits.detach().cpu().tolist()
                )

            CFG.reranker_batch_size = current_bs
            break

        except torch.cuda.OutOfMemoryError:
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

            if current_bs == 1:
                raise

            current_bs = max(1, current_bs // 2)

            print(
                "CUDA OOM detected in reranker. Retrying with "
                f"batch_size={current_bs}"
            )

    reranked = []

    for item, logit in zip(candidates, all_logits):
        x = dict(item)
        x["reranker_logit"] = float(logit)
        x["reranker_prob"] = float(
            torch.sigmoid(torch.tensor(logit))
        )
        reranked.append(x)

    reranked.sort(
        key=lambda x: x["reranker_logit"],
        reverse=True,
    )

    return reranked[:top_k]

In [ ]:
# Cell 34 — End-to-end hybrid retrieval with automatic hard-filter fallback

def retrieve_once(
    query: str,
    plan: Optional[QueryPlan] = None,
) -> dict:
    plan = plan or plan_query(query)

    dense = dense_retrieve(
        semantic_query=plan.semantic_query,
        plan=plan,
        k=CFG.dense_k,
        use_hard_filter=True,
    )

    sparse = bm25_retrieve(
        lexical_query=plan.lexical_query,
        plan=plan,
        k=CFG.bm25_k,
        use_hard_filter=True,
    )

    # Hard-filter fallback if routing is too restrictive.
    if plan.filter_confidence >= CFG.hard_filter_confidence:
        if len(dense) < max(10, CFG.final_leaf_k):
            dense = dense_retrieve(
                semantic_query=plan.semantic_query,
                plan=plan,
                k=CFG.dense_k,
                use_hard_filter=False,
            )
        if len(sparse) < max(10, CFG.final_leaf_k):
            sparse = bm25_retrieve(
                lexical_query=plan.lexical_query,
                plan=plan,
                k=CFG.bm25_k,
                use_hard_filter=False,
            )

    fused = reciprocal_rank_fusion(
        [dense, sparse],
        k_constant=CFG.rrf_constant,
    )

    fused = apply_parent_cap(fused)

    reranked = rerank_candidates(
        query=plan.semantic_query,
        candidates=fused,
        top_k=CFG.final_leaf_k,
    )

    return {
        "query": query,
        "plan": plan,
        "dense": dense,
        "bm25": sparse,
        "fused": fused,
        "reranked": reranked,
    }

In [ ]:
# Cell 35 — Parent/table expansion and evidence-pack construction

GENERATION_ALLOWED_ROLES = {
    "normative_recommendation",
    "implementation_remark",
    "background",
    "table",
    "figure",
    "mixed",
}

def build_evidence_pack(
    retrieval: dict,
) -> list[dict]:
    leaves = retrieval["reranked"]

    parent_scores = {}
    parent_leaf_hits = defaultdict(list)

    for hit in leaves:
        leaf = leaf_lookup[hit["leaf_id"]]
        parent_id = leaf["parent_id"]
        parent_leaf_hits[parent_id].append(hit)

        score = hit["reranker_logit"]
        parent_scores[parent_id] = max(
            score,
            parent_scores.get(parent_id, -1e9),
        )

    ranked_parents = sorted(
        parent_scores,
        key=lambda p: parent_scores[p],
        reverse=True,
    )

    evidence = []
    used_tokens = 0

    for parent_id in ranked_parents:
        parent = parent_lookup[parent_id]

        if parent["content_role"] not in GENERATION_ALLOWED_ROLES:
            continue

        p_tokens = int(parent["token_count"])

        if (
            evidence
            and used_tokens + p_tokens > CFG.max_context_tokens
        ):
            continue

        evidence_id = f"E{len(evidence) + 1}"

        evidence.append({
            "evidence_id": evidence_id,
            "parent_id": parent_id,
            "theme_id": parent["theme_id"],
            "theme_title": parent["theme_title"],
            "module_id": parent["module_id"],
            "module_title": parent["module_title"],
            "section_title": parent["section_title"],
            "content_role": parent["content_role"],
            "page_start": int(parent["page_start"]),
            "page_end": int(parent["page_end"]),
            "table_id": parent.get("table_id"),
            "score": float(parent_scores[parent_id]),
            "leaf_hits": [
                x["leaf_id"]
                for x in parent_leaf_hits[parent_id]
            ],
            "text": parent["text"],
        })

        used_tokens += p_tokens

        if len(evidence) >= CFG.final_parent_k:
            break

    return evidence

In [ ]:
# Cell 36 — Corrective retrieval: rewrite only when evidence quality is weak

def rewrite_query_for_retrieval(
    original_query: str,
    previous_plan: QueryPlan,
    top_hits: list[dict],
) -> QueryPlan:
    if not CFG.llm_model or not CFG.llm_api_key:
        return previous_plan

    hit_summaries = []
    for hit in top_hits[:5]:
        leaf = leaf_lookup[hit["leaf_id"]]
        hit_summaries.append({
            "module_id": leaf["module_id"],
            "section_title": leaf["section_title"],
            "page": leaf["page_start"],
            "text": leaf["text"][:500],
        })

    system = '''
You rewrite clinical retrieval queries for a WHO PEN evidence system.
Return JSON only using the supplied QueryPlan schema.

The previous retrieval was weak.
Rewrite the semantic and lexical queries to improve evidence retrieval.
Do not invent medical facts.
Do not force metadata filters unless you are highly certain.
'''

    user = {
        "original_query": original_query,
        "previous_plan": previous_plan.model_dump(),
        "weak_results": hit_summaries,
        "schema": QueryPlan.model_json_schema(),
    }

    try:
        data = llm_json(
            system,
            json.dumps(user, ensure_ascii=False),
        )
        new_plan = QueryPlan.model_validate(data)
    except Exception:
        return previous_plan

    if new_plan.theme_id not in VALID_THEME_IDS:
        new_plan.theme_id = None
    if new_plan.module_id not in VALID_MODULE_IDS:
        new_plan.module_id = None

    new_plan.content_roles = [
        x for x in new_plan.content_roles
        if x in VALID_CONTENT_ROLES
    ]

    return new_plan

def corrective_retrieve(query: str) -> dict:
    plan = plan_query(query)
    result = retrieve_once(query, plan)

    for _ in range(CFG.corrective_retrieval_loops):
        # Uncalibrated sigmoid(logit) relevance score; calibrate threshold on human gold.
        top_prob = (
            result["reranked"][0]["reranker_prob"]
            if result["reranked"]
            else 0.0
        )

        evidence = build_evidence_pack(result)

        if (
            top_prob >= CFG.min_top_reranker_prob
            and len(evidence) >= 1
        ):
            break

        plan = rewrite_query_for_retrieval(
            original_query=query,
            previous_plan=plan,
            top_hits=result["reranked"],
        )

        result = retrieve_once(query, plan)

    result["evidence"] = build_evidence_pack(result)
    return result

In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 37 — Retrieval smoke tests spanning text, tables, and numeric guidance

SMOKE_QUERIES = [
    "What are the 5A's for brief behavioural counselling?",
    "How is the AUDIT score interpreted for a score of 8 to 15?",
    "What is listed for an AUDIT score of 20 to 40?",
    "What blood pressure readings are used to diagnose raised blood pressure?",
    "What cuff size is listed for an arm circumference of 17 to 22 cm?",
    "What tool is used to estimate 10-year cardiovascular risk?",
    "What are the key steps in tobacco cessation brief intervention?",
    "What is the recommended approach to measuring blood pressure accurately?",
]

for q in SMOKE_QUERIES:
    print("\n" + "=" * 100)
    print("QUERY:", q)
    r = corrective_retrieve(q)
    for rank, hit in enumerate(r["reranked"][:5], start=1):
        leaf = leaf_lookup[hit["leaf_id"]]
        print(
            rank,
            "| p.", leaf["page_start"],
            "|", leaf["module_id"],
            "|", str(leaf["section_title"])[:70],
            "| score=", round(hit["reranker_prob"], 4),
        )
        print(str(leaf["text"])[:500].replace("\n", " "))

print("\nNOTE: 'score' is sigmoid(reranker logit), not a calibrated probability.")
print(f'\n⏱ 37 Retrieval smoke tests: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
# Cell 38 — Structured grounded answer schema

class Claim(BaseModel):
    claim_id: str
    text: str
    evidence_ids: list[str]
    supporting_quote: str = ""

class GroundedAnswer(BaseModel):
    status: Literal[
        "answer",
        "insufficient_evidence",
        "out_of_scope",
        "needs_clinician_review",
    ]
    answer: str
    claims: list[Claim] = Field(default_factory=list)
    safety_note: str = ""

In [ ]:
# Cell 39 — Grounded generation: evidence only, exact evidence IDs

GENERATOR_SYSTEM_PROMPT = '''
You are an evidence-grounded clinical reference assistant built on WHO PEN.

Hard rules:
1. Use ONLY the EVIDENCE blocks provided in this request.
2. Do not use model memory to add clinical facts.
3. Do not invent diagnoses, thresholds, dosages, durations, contraindications or treatment steps.
4. Every factual clinical claim must cite one or more evidence_ids.
5. supporting_quote must be a short exact excerpt from one cited evidence block.
6. If the evidence is incomplete, contradictory, or not specific enough, return insufficient_evidence.
7. For patient-specific questions, do not independently diagnose. Summarize the guideline evidence and mark needs_clinician_review when a clinical judgment step remains.
8. Answer in the same language as the user's question.
9. Return JSON only matching GroundedAnswer.

The evidence, not the language model, is the source of truth.
'''

def serialize_evidence(evidence: list[dict]) -> str:
    blocks = []
    for e in evidence:
        page = (
            str(e["page_start"])
            if e["page_start"] == e["page_end"]
            else f"{e['page_start']}-{e['page_end']}"
        )

        blocks.append(
            f"[{e['evidence_id']}]\n"
            f"Document: {CFG.document_title}\n"
            f"Module: {e['module_id']} — {e['module_title']}\n"
            f"Section: {e['section_title']}\n"
            f"PDF page(s): {page}\n"
            f"Content role: {e['content_role']}\n"
            f"TEXT:\n{e['text']}"
        )

    return "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(blocks)

def generate_grounded_answer(
    query: str,
    evidence: list[dict],
    feedback: str = "",
) -> GroundedAnswer:
    if not evidence:
        return GroundedAnswer(
            status="insufficient_evidence",
            answer="The retrieved WHO PEN evidence is insufficient to answer reliably.",
            safety_note="No eligible evidence block passed retrieval and content-role gates.",
        )

    user_payload = (
        f"USER QUERY:\n{query}\n\n"
        f"EVIDENCE:\n{serialize_evidence(evidence)}\n\n"
        f"PREVIOUS VERIFICATION FEEDBACK:\n{feedback or 'None'}\n\n"
        f"JSON SCHEMA:\n"
        f"{json.dumps(GroundedAnswer.model_json_schema(), ensure_ascii=False)}"
    )

    data = llm_json(
        GENERATOR_SYSTEM_PROMPT,
        user_payload,
        model=CFG.llm_model,
    )

    return GroundedAnswer.model_validate(data)

In [ ]:
# Cell 40 — Deterministic citation and numeric safety guards

NUMERIC_PATTERN = re.compile(
    r"(?<!\w)"
    r"(?:\d+(?:\.\d+)?(?:\s*[–-]\s*\d+(?:\.\d+)?)?"
    r"(?:\s*(?:mmHg|mg/dL|mmol/L|mg|µg|mcg|mL|ml|cm|kg|%))?"
    r"|\d+\s*/\s*\d+)"
    r"(?!\w)",
    re.I,
)

def normalized_numeric_tokens(text: str) -> set[str]:
    tokens = set()
    for match in NUMERIC_PATTERN.findall(text or ""):
        t = re.sub(r"\s+", "", match.lower())
        t = t.replace("-", "–")
        tokens.add(t)
    return tokens

def deterministic_verify(
    answer: GroundedAnswer,
    evidence: list[dict],
) -> dict:
    evidence_map = {e["evidence_id"]: e for e in evidence}
    errors = []

    for claim in answer.claims:
        if not claim.evidence_ids:
            errors.append(
                f"{claim.claim_id}: no evidence_ids"
            )
            continue

        missing = [
            eid for eid in claim.evidence_ids
            if eid not in evidence_map
        ]
        if missing:
            errors.append(
                f"{claim.claim_id}: invalid evidence IDs {missing}"
            )
            continue

        cited_text = "\n".join(
            evidence_map[eid]["text"]
            for eid in claim.evidence_ids
        )

        # Quote must be verbatim if supplied.
        if claim.supporting_quote:
            quote_norm = re.sub(
                r"\s+", " ", claim.supporting_quote.strip()
            )
            cited_norm = re.sub(
                r"\s+", " ", cited_text
            )
            if quote_norm not in cited_norm:
                errors.append(
                    f"{claim.claim_id}: supporting_quote is not verbatim in cited evidence"
                )

        claim_numbers = normalized_numeric_tokens(claim.text)
        evidence_numbers = normalized_numeric_tokens(cited_text)

        unsupported_numbers = sorted(
            claim_numbers - evidence_numbers
        )

        if unsupported_numbers:
            errors.append(
                f"{claim.claim_id}: unsupported numeric tokens {unsupported_numbers}"
            )

    return {
        "passed": len(errors) == 0,
        "errors": errors,
    }

In [ ]:
# Cell 41 — Independent LLM entailment verifier

class ClaimVerdict(BaseModel):
    claim_id: str
    verdict: Literal["supported", "partial", "unsupported"]
    reason: str

class VerificationReport(BaseModel):
    verdicts: list[ClaimVerdict]
    overall_pass: bool

VERIFIER_SYSTEM = '''
You are an independent evidence verifier for a clinical RAG system.

For each claim:
- supported: the cited evidence directly supports the full claim.
- partial: only part of the claim is supported, or the wording is stronger than the evidence.
- unsupported: the cited evidence does not support the claim.

Do not use external medical knowledge.
Evaluate only claim versus cited evidence.
Return JSON only matching VerificationReport.
'''

def llm_verify_claims(
    answer: GroundedAnswer,
    evidence: list[dict],
) -> VerificationReport:
    if not answer.claims:
        return VerificationReport(
            verdicts=[],
            overall_pass=(answer.status != "answer"),
        )

    evidence_map = {e["evidence_id"]: e for e in evidence}

    payload_claims = []
    for claim in answer.claims:
        cited = [
            {
                "evidence_id": eid,
                "text": evidence_map[eid]["text"],
            }
            for eid in claim.evidence_ids
            if eid in evidence_map
        ]

        payload_claims.append({
            "claim_id": claim.claim_id,
            "claim": claim.text,
            "cited_evidence": cited,
        })

    model = CFG.verifier_model or CFG.llm_model

    data = llm_json(
        VERIFIER_SYSTEM,
        json.dumps({
            "claims": payload_claims,
            "schema": VerificationReport.model_json_schema(),
        }, ensure_ascii=False),
        model=model,
    )

    return VerificationReport.model_validate(data)

In [ ]:
# Cell 42 — Full verified answer pipeline with maximum two refinement attempts

def render_citations(
    answer: GroundedAnswer,
    evidence: list[dict],
) -> str:
    evidence_map = {e["evidence_id"]: e for e in evidence}
    lines = [answer.answer.strip()]

    if answer.claims:
        lines.append("\nEvidence trace:")
        for claim in answer.claims:
            refs = []
            for eid in claim.evidence_ids:
                if eid not in evidence_map:
                    continue
                e = evidence_map[eid]
                page = (
                    str(e["page_start"])
                    if e["page_start"] == e["page_end"]
                    else f"{e['page_start']}-{e['page_end']}"
                )
                refs.append(
                    f"{eid}: Module {e['module_id']}, "
                    f"{e['section_title'] or 'section'}, PDF p.{page}"
                )
            if refs:
                lines.append(
                    f"- {claim.claim_id}: " + " | ".join(refs)
                )

    if answer.safety_note:
        lines.append("\nSafety: " + answer.safety_note.strip())

    return "\n".join(lines)

def ask_medical_rag(
    query: str,
    max_refinements: int = 2,
) -> dict:
    retrieval = corrective_retrieve(query)
    evidence = retrieval["evidence"]

    if not evidence:
        answer = GroundedAnswer(
            status="insufficient_evidence",
            answer="The WHO PEN evidence retrieved is insufficient to answer reliably.",
            safety_note="No generation was attempted.",
        )
        return {
            "query": query,
            "retrieval": retrieval,
            "answer": answer,
            "verification": None,
            "rendered": render_citations(answer, evidence),
        }

    feedback = ""
    final_det = None
    final_llm_verification = None
    answer = None

    for attempt in range(max_refinements + 1):
        answer = generate_grounded_answer(
            query=query,
            evidence=evidence,
            feedback=feedback,
        )

        final_det = deterministic_verify(
            answer,
            evidence,
        )

        if not final_det["passed"]:
            feedback = (
                "Deterministic verifier failed:\n- "
                + "\n- ".join(final_det["errors"])
            )
            continue

        final_llm_verification = llm_verify_claims(
            answer,
            evidence,
        )

        if final_llm_verification.overall_pass and all(
            v.verdict == "supported"
            for v in final_llm_verification.verdicts
        ):
            break

        feedback = "Entailment verifier rejected claims:\n" + "\n".join(
            f"{v.claim_id}: {v.verdict} — {v.reason}"
            for v in final_llm_verification.verdicts
            if v.verdict != "supported"
        )

    # Fail closed after refinement budget.
    if (
        not final_det
        or not final_det["passed"]
        or (
            final_llm_verification
            and not final_llm_verification.overall_pass
        )
    ):
        answer = GroundedAnswer(
            status="insufficient_evidence",
            answer=(
                "The system could not produce an answer that passed "
                "citation, numeric, and entailment verification."
            ),
            safety_note="Fail-closed safety gate triggered.",
        )

    return {
        "query": query,
        "retrieval": retrieval,
        "answer": answer,
        "deterministic_verification": final_det,
        "llm_verification": (
            final_llm_verification.model_dump()
            if final_llm_verification else None
        ),
        "rendered": render_citations(answer, evidence),
    }

In [ ]:
# Cell 43 — Production query example

# Before running this cell, set:
# os.environ["LLM_API_KEY"] = "..."
# os.environ["LLM_MODEL"] = "..."
# CFG.llm_api_key = os.environ["LLM_API_KEY"]
# CFG.llm_model = os.environ["LLM_MODEL"]
# CFG.verifier_model = os.getenv("VERIFIER_MODEL", CFG.llm_model)

# result = ask_medical_rag(
#     "According to WHO PEN, how is an AUDIT score of 16–19 managed?"
# )
# print(result["rendered"])

In [ ]:
# Cell 44 — Bootstrap source regression set: exact phrases located in the full PDF

# raw_page_texts may have been released by Cell 24B to save RAM. Recover it cheaply
# from the native PDF text layer if required; this does NOT rerun Docling.
if "raw_page_texts" not in globals():
    _pdf_eval = fitz.open(str(PDF_PATH))
    raw_page_texts = {
        p: (_pdf_eval[p - 1].get_text("text") or "")
        for p in range(1, len(_pdf_eval) + 1)
    }
    _pdf_eval.close()
    del _pdf_eval
    print("✓ Reloaded native page text for bootstrap evaluation:", len(raw_page_texts))


BOOTSTRAP_CASES = [
    {
        "id": "B01",
        "question": "What are the five steps in the 5A's brief intervention?",
        "phrase": "Ask, Advise, Assess, Assist, Arrange",
    },
    {
        "id": "B02",
        "question": "What are the five components of the 5R's intervention?",
        "phrase": "Relevance, Risks, Rewards, Roadblocks, Repetition",
    },
    {
        "id": "B03",
        "question": "What is the AUDIT screening tool used for?",
        "phrase": "AUDIT is a 10-item screening tool",
    },
    {
        "id": "B04",
        "question": "What type of intervention is listed for an AUDIT score of 8 to 15?",
        "phrase": "8–15 Simple advice",
    },
    {
        "id": "B05",
        "question": "What is listed for an AUDIT score of 20 to 40?",
        "phrase": "20–40 Referral to specialist",
    },
    {
        "id": "B06",
        "question": "What blood pressure criteria are stated for diagnosing raised BP?",
        "phrase": "Raised BP is diagnosed if there are two raised",
    },
    {
        "id": "B07",
        "question": "What cuff size is listed for an arm circumference of 17 to 22 cm?",
        "phrase": "17–22 Small",
    },
    {
        "id": "B08",
        "question": "What tool is used to estimate 10-year cardiovascular risk?",
        "phrase": "risk prediction chart",
    },
]

def norm_for_phrase_match(text: str) -> str:
    text = normalize_medical_text(text)
    text = text.replace("-", "–")
    text = re.sub(r"\s+", " ", text).lower()
    return text

bootstrap_gold = []

for case in BOOTSTRAP_CASES:
    needle = norm_for_phrase_match(case["phrase"])
    pages = [
        p for p, text in raw_page_texts.items()
        if needle in norm_for_phrase_match(text)
    ]

    bootstrap_gold.append({
        **case,
        "gold_pages": pages,
    })

bootstrap_gold_df = pd.DataFrame(bootstrap_gold)
display(bootstrap_gold_df)

In [ ]:
import time as _stage_time
_stage_t0 = _stage_time.perf_counter()
# Cell 45 — Bootstrap retrieval regression metrics

def bootstrap_retrieval_eval(
    gold_df: pd.DataFrame,
    k: int = 10,
) -> pd.DataFrame:
    rows = []

    for item in tqdm(
        gold_df.to_dict("records"),
        desc="Bootstrap retrieval eval"
    ):
        result = corrective_retrieve(item["question"])
        hits = result["reranked"][:k]

        retrieved_pages = []
        for hit in hits:
            leaf = leaf_lookup[hit["leaf_id"]]
            retrieved_pages.extend(
                range(
                    int(leaf["page_start"]),
                    int(leaf["page_end"]) + 1
                )
            )

        gold_pages = set(item["gold_pages"])
        first_match_rank = None

        for rank, hit in enumerate(hits, start=1):
            leaf = leaf_lookup[hit["leaf_id"]]
            hit_pages = set(
                range(
                    int(leaf["page_start"]),
                    int(leaf["page_end"]) + 1
                )
            )
            if gold_pages & hit_pages:
                first_match_rank = rank
                break

        rows.append({
            "id": item["id"],
            "question": item["question"],
            "gold_pages": item["gold_pages"],
            "hit_at_k": first_match_rank is not None,
            "first_match_rank": first_match_rank,
            "reciprocal_rank": (
                1.0 / first_match_rank
                if first_match_rank else 0.0
            ),
        })

    return pd.DataFrame(rows)

bootstrap_eval_df = bootstrap_retrieval_eval(
    bootstrap_gold_df,
    k=10,
)

display(bootstrap_eval_df)

print("HitRate@10:", bootstrap_eval_df["hit_at_k"].mean())
print("MRR:", bootstrap_eval_df["reciprocal_rank"].mean())

bootstrap_eval_df.to_csv(
    CFG.root / "eval" / "bootstrap_retrieval_eval.csv",
    index=False,
)
print(f'\n⏱ 45 Bootstrap retrieval evaluation: {(_stage_time.perf_counter() - _stage_t0)/60:.2f} min')


In [ ]:
# Cell 46 — Human gold dataset schema for production retrieval evaluation

HUMAN_GOLD_PATH = CFG.root / "eval" / "human_gold.jsonl"

HUMAN_GOLD_TEMPLATE = {
    "question_id": "Q001",
    "question": "Clinical question",
    "relevant_leaf_ids": ["leaf_..."],
    "relevant_parent_ids": ["parent_..."],
    "reference_answer": "Clinician-reviewed reference answer",
    "notes": "Why these evidence nodes are relevant",
}

if not HUMAN_GOLD_PATH.exists():
    with HUMAN_GOLD_PATH.open("w", encoding="utf-8") as f:
        f.write(json.dumps(HUMAN_GOLD_TEMPLATE, ensure_ascii=False) + "\n")

print("Human gold file:", HUMAN_GOLD_PATH)
print(
    "Replace the template with clinician-reviewed labels before production acceptance."
)

In [ ]:
# Cell 47 — Exact ID-based retrieval metrics: Precision@K, Recall@K, MRR, nDCG

def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    if k == 0:
        return 0.0
    return sum(x in relevant for x in retrieved_k) / k

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return 0.0
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & set(relevant)) / len(set(relevant))

def reciprocal_rank(retrieved, relevant):
    relevant = set(relevant)
    for rank, x in enumerate(retrieved, start=1):
        if x in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    dcg = 0.0
    for i, x in enumerate(retrieved[:k], start=1):
        rel = 1.0 if x in relevant else 0.0
        dcg += rel / math.log2(i + 1)

    ideal_hits = min(k, len(relevant))
    idcg = sum(
        1.0 / math.log2(i + 1)
        for i in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0

def load_human_gold(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def evaluate_human_gold(gold_rows: list[dict]) -> pd.DataFrame:
    metrics = []

    for item in tqdm(gold_rows, desc="Human gold evaluation"):
        result = corrective_retrieve(item["question"])
        retrieved_leaf_ids = [
            x["leaf_id"]
            for x in result["reranked"]
        ]
        relevant = set(item["relevant_leaf_ids"])

        metrics.append({
            "question_id": item["question_id"],
            "precision@5": precision_at_k(
                retrieved_leaf_ids, relevant, 5
            ),
            "recall@10": recall_at_k(
                retrieved_leaf_ids, relevant, 10
            ),
            "mrr": reciprocal_rank(
                retrieved_leaf_ids, relevant
            ),
            "ndcg@10": ndcg_at_k(
                retrieved_leaf_ids, relevant, 10
            ),
        })

    return pd.DataFrame(metrics)

# Run only after replacing the template with real labels:
# gold = load_human_gold(HUMAN_GOLD_PATH)
# gold = [x for x in gold if not any(v == "leaf_..." for v in x["relevant_leaf_ids"])]
# human_eval_df = evaluate_human_gold(gold)
# display(human_eval_df)
# print(human_eval_df.mean(numeric_only=True))

In [ ]:

# Cell 48 — Resumable, rate-limit-safe RAGAS evaluation

import asyncio
import math
import time

def _exception_chain_text(exc: BaseException) -> str:
    parts, seen = [], set()
    cur = exc
    while cur is not None and id(cur) not in seen:
        seen.add(id(cur))
        parts.append(f"{type(cur).__name__}: {cur}")
        cur = cur.__cause__ or cur.__context__
    return " | ".join(parts)

def _is_rate_limit_error(exc: BaseException) -> bool:
    text = _exception_chain_text(exc).lower()
    return any(x in text for x in [
        "429", "too many requests", "rate limit", "rate_limit", "quota"
    ])

def _retry_after_seconds(exc: BaseException, fallback: float = 30.0) -> float:
    cur, seen = exc, set()
    while cur is not None and id(cur) not in seen:
        seen.add(id(cur))
        response = getattr(cur, "response", None)
        headers = getattr(response, "headers", None)
        if headers:
            value = headers.get("retry-after") or headers.get("Retry-After")
            if value:
                try:
                    return max(float(value), 1.0)
                except Exception:
                    pass
        cur = cur.__cause__ or cur.__context__
    return float(fallback)

async def run_ragas_eval(
    gold_rows: list[dict],
    max_questions: int = None,
    *,
    resume: bool = True,
    retry_failed: bool = True,
    delay_between_metrics: float = 6.0,
    delay_between_questions: float = 12.0,
    max_rate_limit_retries: int = 1,
    checkpoint_name: str = "ragas_progress.csv",
):
    require_llm_config()

    from openai import AsyncOpenAI
    from ragas import SingleTurnSample
    from ragas.llms import llm_factory

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"ragas.*")
        from ragas.metrics import (
            LLMContextPrecisionWithReference,
            LLMContextRecall,
            Faithfulness,
        )
    metric_import_path = "ragas.metrics legacy API (0.4.3-compatible)"

    client = AsyncOpenAI(
        api_key=CFG.llm_api_key,
        base_url=CFG.llm_base_url,
        timeout=CFG.request_timeout,
        max_retries=1,
    )
    evaluator_llm = llm_factory(
        CFG.verifier_model or CFG.llm_model,
        provider="openai",
        client=client,
    )

    metrics = [
        ("context_precision", LLMContextPrecisionWithReference(llm=evaluator_llm)),
        ("context_recall", LLMContextRecall(llm=evaluator_llm)),
        ("faithfulness", Faithfulness(llm=evaluator_llm)),
    ]

    eval_dir = CFG.root / "eval"
    eval_dir.mkdir(parents=True, exist_ok=True)
    progress_path = eval_dir / checkpoint_name

    selected = gold_rows[:max_questions] if max_questions else list(gold_rows)
    row_map, done_ids = {}, set()

    if resume and progress_path.exists():
        try:
            prev = pd.read_csv(progress_path)
            for row in prev.to_dict("records"):
                qid = str(row.get("question_id"))
                row_map[qid] = row
                ok = str(row.get("evaluation_status", "")).lower() == "ok"
                if ok or not retry_failed:
                    done_ids.add(qid)
            print(f"↻ Resuming RAGAS: {len(done_ids)} completed rows found.")
        except Exception as e:
            print("⚠ Previous RAGAS progress could not be read:", e)

    async def score_metric(metric, sample, metric_name):
        for attempt in range(max_rate_limit_retries + 1):
            try:
                return float(await metric.single_turn_ascore(sample))
            except Exception as exc:
                if not _is_rate_limit_error(exc) or attempt >= max_rate_limit_retries:
                    raise
                wait_s = _retry_after_seconds(exc)
                print(f"⏳ {metric_name}: rate-limited; retrying after {wait_s:.0f}s")
                await asyncio.sleep(wait_s)

    blocked = False

    for item in tqdm(selected, desc="RAGAS eval"):
        qid = str(item["question_id"])
        if qid in done_ids:
            continue

        row = {
            "question_id": qid,
            "context_precision": math.nan,
            "context_recall": math.nan,
            "faithfulness": math.nan,
            "answer_status": None,
            "evaluation_status": "started",
            "error": "",
            "metric_import_path": metric_import_path,
        }

        try:
            app = ask_medical_rag(item["question"])
            contexts = [x["text"] for x in app["retrieval"]["evidence"]]
            row["answer_status"] = app["answer"].status

            sample = SingleTurnSample(
                user_input=item["question"],
                response=app["answer"].answer,
                reference=item["reference_answer"],
                retrieved_contexts=contexts,
            )

            for metric_name, metric in metrics:
                row[metric_name] = await score_metric(metric, sample, metric_name)
                if delay_between_metrics > 0:
                    await asyncio.sleep(delay_between_metrics)

            row["evaluation_status"] = "ok"

        except Exception as exc:
            row["error"] = _exception_chain_text(exc)[:1500]
            if _is_rate_limit_error(exc):
                row["evaluation_status"] = "blocked_rate_limit"
                blocked = True
            else:
                row["evaluation_status"] = "error"

        row_map[qid] = row
        pd.DataFrame(row_map.values()).to_csv(progress_path, index=False)

        if blocked:
            print("\\n⏸ Provider quota/rate-limit detected.")
            print("✅ Partial RAGAS progress saved:", progress_path)
            print("➡️ Re-run later with resume=True; completed rows will be reused.")
            break

        if delay_between_questions > 0:
            await asyncio.sleep(delay_between_questions)

    result_df = pd.DataFrame(row_map.values())
    if len(result_df):
        result_df = result_df.sort_values("question_id").reset_index(drop=True)

    ok_df = (
        result_df[result_df["evaluation_status"].eq("ok")].copy()
        if len(result_df) else result_df
    )

    print("\\n" + "=" * 88)
    print("RAGAS EVALUATION STATUS")
    print("=" * 88)
    print("Requested questions :", len(selected))
    print("Saved rows          :", len(result_df))
    print("Completed OK        :", len(ok_df))

    if len(result_df):
        print(result_df["evaluation_status"].value_counts(dropna=False).to_string())

    if len(ok_df):
        display(
            ok_df[
                ["context_precision", "context_recall", "faithfulness"]
            ].mean().to_frame("mean")
        )
    else:
        print("No completed RAGAS rows. Production RAGAS gate remains PENDING.")

    print("=" * 88)
    return result_df

# Pilot:
# ragas_10_df = await run_ragas_eval(gold100_df.to_dict("records"), max_questions=10)


In [ ]:

# Cell 49 — Production acceptance gates (fail/pending closed)

PRODUCTION_GATES = {
    "recall@10": 0.85,
    "precision@5": 0.65,
    "context_precision": 0.80,
    "context_recall": 0.80,
    "faithfulness": 0.80,
    "critical_numeric_mismatches": 0,
    "invalid_citations": 0,
    "unsupported_clinical_claims": 0,
}

def check_production_gates(
    retrieval_metrics: Optional[pd.DataFrame] = None,
    ragas_metrics: Optional[pd.DataFrame] = None,
    critical_numeric_mismatches: int = 0,
    invalid_citations: int = 0,
    unsupported_clinical_claims: int = 0,
    *,
    require_retrieval: bool = True,
    require_ragas: bool = True,
):
    failures, pending = [], []

    if retrieval_metrics is None or not len(retrieval_metrics):
        if require_retrieval:
            pending.append("retrieval_metrics_missing")
    else:
        means = retrieval_metrics.mean(numeric_only=True)
        for metric in ["recall@10", "precision@5"]:
            if metric not in means:
                pending.append(f"{metric}_missing")
            elif means[metric] < PRODUCTION_GATES[metric]:
                failures.append(
                    f"{metric}={means[metric]:.3f} < {PRODUCTION_GATES[metric]}"
                )

    if ragas_metrics is None or not len(ragas_metrics):
        if require_ragas:
            pending.append("ragas_metrics_missing")
    else:
        work = ragas_metrics.copy()
        if "evaluation_status" in work.columns:
            work = work[work["evaluation_status"].eq("ok")]

        if not len(work):
            if require_ragas:
                pending.append("ragas_no_completed_rows")
        else:
            means = work.mean(numeric_only=True)
            for metric in ["context_precision", "context_recall", "faithfulness"]:
                if metric not in means:
                    pending.append(f"{metric}_missing")
                elif means[metric] < PRODUCTION_GATES[metric]:
                    failures.append(
                        f"{metric}={means[metric]:.3f} < {PRODUCTION_GATES[metric]}"
                    )

    if critical_numeric_mismatches != 0:
        failures.append(f"critical_numeric_mismatches={critical_numeric_mismatches}")
    if invalid_citations != 0:
        failures.append(f"invalid_citations={invalid_citations}")
    if unsupported_clinical_claims != 0:
        failures.append(f"unsupported_clinical_claims={unsupported_clinical_claims}")

    status = "failed" if failures else ("pending" if pending else "passed")

    return {
        "status": status,
        "passed": status == "passed",
        "failures": failures,
        "pending": pending,
    }

print(json.dumps(PRODUCTION_GATES, indent=2))
print("\\nSafety rule: missing required evaluation is PENDING, never silently PASS.")


In [ ]:

# Cell 50 — Save complete run manifest for reproducibility (SECRET-SAFE)

import importlib.metadata as metadata

important_packages = [
    "docling",
    "qdrant-client",
    "sentence-transformers",
    "transformers",
    "rank-bm25",
    "pymupdf",
    "ragas",
    "langchain-community",
    "gradio",
    "openai",
]

versions = {}
for pkg in important_packages:
    try:
        versions[pkg] = metadata.version(pkg)
    except Exception:
        versions[pkg] = "unknown"

def _redact_config(config_dict: dict) -> dict:
    secret_markers = (
        "api_key", "apikey", "secret", "password",
        "access_token", "refresh_token", "private_key",
    )
    safe = {}
    for key, value in config_dict.items():
        low = str(key).lower()
        if any(marker in low for marker in secret_markers):
            safe[f"{key}_configured"] = bool(value)
        else:
            safe[key] = value
    return safe

safe_cfg = _redact_config(asdict(CFG))
safe_cfg["root"] = str(CFG.root)

run_manifest = {
    "notebook_version": "v7-production-gui",
    "source": source_registry,
    "config": safe_cfg,
    "package_versions": versions,
    "device": DEVICE,
    "parents": len(parents_df),
    "leaves": len(leaves_df),
    "tables": len(table_records),
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
}

run_manifest_path = CFG.root / "logs" / "run_manifest.json"
run_manifest_path.write_text(
    json.dumps(run_manifest, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

_manifest_text = run_manifest_path.read_text(encoding="utf-8")
for forbidden in ["gsk_", "sk-proj-", '"llm_api_key":']:
    if forbidden in _manifest_text:
        raise RuntimeError(f"Secret-safety check failed: {forbidden!r}")

print("✅ Secret-safe run manifest written:")
print(run_manifest_path)
print(json.dumps(run_manifest, indent=2, ensure_ascii=False, default=str)[:5000])


In [ ]:
# Cell 50B — Runtime summary from this notebook session

timing_rows = []

for name, seconds in sorted(
    STAGE_TIMINGS.items(),
    key=lambda x: x[1],
    reverse=True,
):
    timing_rows.append({
        "stage": name,
        "minutes": seconds / 60.0,
        "seconds": seconds,
    })

runtime_df = pd.DataFrame(timing_rows)

if len(runtime_df):
    display(runtime_df)
else:
    print(
        "STAGE_TIMINGS is empty because some earlier cells use direct wall-clock prints. "
        "Use the printed ⏱ lines above as the source of truth for this run."
    )

print("\nCurrent selected batch sizes")
print("Embedding batch :", CFG.embedding_batch_size)
print("Reranker batch  :", CFG.reranker_batch_size)

In [ ]:

# Cell 50C — Save v7 checkpoint safely, then reopen Qdrant

from google.colab import drive
import shutil, gc, time

drive.mount("/content/drive", force_remount=False)

DRIVE_PROJECT_ROOT = Path(os.getenv("WHO_PEN_DRIVE_ROOT", "/content/drive/MyDrive/WHO_PEN_RAG"))
CHECKPOINT_DIR = DRIVE_PROJECT_ROOT / "who_pen_medical_rag_v7_gui_checkpoint"
TEMP_CHECKPOINT_DIR = CHECKPOINT_DIR.with_name(CHECKPOINT_DIR.name + "_tmp")

if "qdrant" in globals():
    try:
        qdrant.close()
    except Exception:
        pass

gc.collect()
time.sleep(1)

if TEMP_CHECKPOINT_DIR.exists():
    shutil.rmtree(TEMP_CHECKPOINT_DIR)

shutil.copytree(CFG.root, TEMP_CHECKPOINT_DIR)

important = [
    "source/source_manifest.json",
    "cache/docling_document.json",
    "canonical/module_ranges.parquet",
    "canonical/page_meta.json",
    "canonical/elements.parquet",
    "canonical/tables.jsonl",
    "chunks/parents.parquet",
    "chunks/leaves.parquet",
    "index/leaf_embeddings.npy",
    "index/leaf_embeddings.meta.json",
    "index/bm25.joblib",
    "index/qdrant",
    "index/qdrant.meta.json",
    "logs/run_manifest.json",
]

all_ok = True
for rel in important:
    ok = (TEMP_CHECKPOINT_DIR / rel).exists()
    print("✅" if ok else "❌", rel)
    all_ok &= ok

assert all_ok, "Temporary checkpoint incomplete. Previous checkpoint was NOT replaced."

manifest_text = (TEMP_CHECKPOINT_DIR / "logs" / "run_manifest.json").read_text(encoding="utf-8")
assert '"llm_api_key":' not in manifest_text

if CHECKPOINT_DIR.exists():
    shutil.rmtree(CHECKPOINT_DIR)

TEMP_CHECKPOINT_DIR.rename(CHECKPOINT_DIR)
print("✅ v7 checkpoint complete:", CHECKPOINT_DIR)

qdrant = QdrantClient(path=str(QDRANT_DIR))
assert qdrant.collection_exists(COLLECTION)
assert (qdrant.get_collection(COLLECTION).points_count or 0) == len(leaves_df)
print("✅ Qdrant reopened; runtime can continue safely.")



# Production Demo GUI — Gradio Blocks

Evidence-first clinical UI:
- calm professional design,
- subtle CSS animations,
- progress stages,
- grounded answer,
- citations and expandable evidence,
- developer retrieval/verification trace.

The GUI calls the same production pipeline defined above.


In [ ]:

# Cell 51 — Professional Gradio GUI helpers

import html
import traceback
import gradio as gr

GUI_CSS = r"""
:root {
  --pen-bg: #f5f7f7;
  --pen-card: rgba(255,255,255,0.93);
  --pen-border: #dfe7e7;
  --pen-text: #1f2a2a;
  --pen-muted: #667575;
  --pen-accent: #2f6f70;
  --pen-accent-soft: #e8f2f2;
  --pen-success: #28765a;
  --pen-warning: #a46722;
  --pen-danger: #a34545;
}
.gradio-container {
  background:
    radial-gradient(circle at 8% 0%, rgba(47,111,112,.08), transparent 28%),
    radial-gradient(circle at 92% 4%, rgba(74,120,95,.06), transparent 24%),
    var(--pen-bg) !important;
}
.pen-shell { max-width: 1180px; margin: 0 auto; }
.pen-hero {
  padding: 24px 26px; border: 1px solid var(--pen-border);
  border-radius: 24px;
  background: linear-gradient(135deg, rgba(255,255,255,.96), rgba(239,247,246,.96));
  box-shadow: 0 18px 55px rgba(31,42,42,.08);
  animation: penFadeUp .55s ease-out both;
}
.pen-title { font-size: 30px; font-weight: 750; letter-spacing: -.02em; margin: 8px 0; }
.pen-subtitle { color: var(--pen-muted); font-size: 15px; line-height: 1.65; }
.pen-badge {
  display: inline-flex; align-items: center; gap: 8px;
  padding: 7px 11px; border-radius: 999px;
  background: var(--pen-accent-soft); color: var(--pen-accent);
  border: 1px solid #d4e7e6; font-size: 12px; font-weight: 700;
}
.pen-dot {
  width: 8px; height: 8px; border-radius: 50%;
  background: var(--pen-accent); animation: penPulse 2s infinite;
}
.pen-card {
  border: 1px solid var(--pen-border) !important;
  border-radius: 20px !important;
  background: var(--pen-card) !important;
  box-shadow: 0 10px 34px rgba(31,42,42,.055) !important;
  animation: penFadeUp .5s ease-out both;
}
.pen-status {
  padding: 11px 14px; border-radius: 14px; font-size: 13px;
  font-weight: 650; border: 1px solid var(--pen-border); background: #fff;
}
.pen-status.ok { color: var(--pen-success); background: #f0f8f4; border-color: #d4eadf; }
.pen-status.warn { color: var(--pen-warning); background: #fff7ed; border-color: #f1dfc9; }
.pen-status.fail { color: var(--pen-danger); background: #fff3f3; border-color: #ecd3d3; }
.pen-evidence {
  margin: 10px 0; padding: 14px 16px; border: 1px solid var(--pen-border);
  border-radius: 15px; background: #fbfdfd;
}
.pen-evidence summary { cursor: pointer; font-weight: 700; color: var(--pen-accent); }
.pen-evidence pre {
  white-space: pre-wrap; word-break: break-word;
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
  font-size: 12px; line-height: 1.65; color: #344;
}
button.primary {
  background: var(--pen-accent) !important; border: none !important;
  transition: transform .18s ease, box-shadow .18s ease !important;
}
button.primary:hover {
  transform: translateY(-1px);
  box-shadow: 0 9px 22px rgba(47,111,112,.18) !important;
}
@keyframes penPulse {
  0% { box-shadow: 0 0 0 0 rgba(47,111,112,.35); }
  70% { box-shadow: 0 0 0 8px rgba(47,111,112,0); }
  100% { box-shadow: 0 0 0 0 rgba(47,111,112,0); }
}
@keyframes penFadeUp {
  from { opacity: 0; transform: translateY(7px); }
  to { opacity: 1; transform: translateY(0); }
}
"""

def _gui_status_html(status: str, verified: bool, elapsed: float) -> str:
    if status == "answer" and verified:
        cls, label = "ok", "Verified grounded answer"
    elif status in {"insufficient_evidence", "needs_clinician_review"}:
        cls, label = "warn", status.replace("_", " ").title()
    else:
        cls, label = "fail", (status or "Unable to complete").replace("_", " ").title()
    return f'<div class="pen-status {cls}">{html.escape(label)} • {elapsed:.2f}s</div>'

def _gui_sources(app: dict) -> str:
    answer = app["answer"]
    evidence = app["retrieval"].get("evidence", [])
    emap = {e["evidence_id"]: e for e in evidence}

    ids = []
    for claim in answer.claims:
        for eid in claim.evidence_ids:
            if eid in emap and eid not in ids:
                ids.append(eid)

    if not ids:
        ids = [e["evidence_id"] for e in evidence[:5]]
    if not ids:
        return "_No eligible evidence was retrieved._"

    lines = []
    for i, eid in enumerate(ids, 1):
        e = emap[eid]
        page = str(e["page_start"]) if e["page_start"] == e["page_end"] else f'{e["page_start"]}–{e["page_end"]}'
        lines.append(
            f"**{i}. {eid}**  \n"
            f"Module **{e['module_id']} — {e['module_title']}** · "
            f"PDF p. **{page}** · {e['section_title'] or 'Section'}"
        )
    return "\\n\\n".join(lines)

def _gui_evidence(app: dict) -> str:
    evidence = app["retrieval"].get("evidence", [])
    if not evidence:
        return '<div class="pen-status warn">No evidence blocks available.</div>'

    blocks = []
    for i, e in enumerate(evidence, 1):
        page = str(e["page_start"]) if e["page_start"] == e["page_end"] else f'{e["page_start"]}–{e["page_end"]}'
        title = f'{i}. {e["evidence_id"]} · Module {e["module_id"]} · PDF p.{page} · {e["section_title"] or "Section"}'
        blocks.append(
            '<details class="pen-evidence">'
            f'<summary>{html.escape(title)}</summary>'
            f'<pre>{html.escape(str(e["text"]))}</pre>'
            '</details>'
        )
    return "".join(blocks)

def _gui_dev_trace(app: dict, elapsed: float) -> dict:
    retrieval = app.get("retrieval", {})
    plan = retrieval.get("plan")
    if hasattr(plan, "model_dump"):
        plan = plan.model_dump()

    hits = []
    for hit in retrieval.get("reranked", [])[:10]:
        leaf = leaf_lookup.get(hit["leaf_id"], {})
        hits.append({
            "leaf_id": hit.get("leaf_id"),
            "module_id": leaf.get("module_id"),
            "page_start": leaf.get("page_start"),
            "content_role": leaf.get("content_role"),
            "section_title": leaf.get("section_title"),
            "reranker_logit": hit.get("reranker_logit"),
            "reranker_sigmoid": hit.get("reranker_prob"),
        })

    return {
        "latency_seconds": round(elapsed, 3),
        "query_plan": plan,
        "retrieval_counts": {
            "dense": len(retrieval.get("dense", [])),
            "bm25": len(retrieval.get("bm25", [])),
            "fused": len(retrieval.get("fused", [])),
            "reranked": len(retrieval.get("reranked", [])),
            "evidence_parents": len(retrieval.get("evidence", [])),
        },
        "top_reranked_leaves": hits,
        "deterministic_verification": app.get("deterministic_verification"),
        "llm_verification": app.get("llm_verification"),
    }

def run_gui_query(query: str, progress=gr.Progress()):
    query = (query or "").strip()
    if not query:
        return (
            '<div class="pen-status warn">Enter a clinical reference question first.</div>',
            "### No question submitted",
            "_No sources yet._",
            '<div class="pen-status warn">No evidence yet.</div>',
            {},
        )

    t0 = time.perf_counter()
    try:
        require_llm_config()
        progress(0.08, desc="Planning query")
        progress(0.18, desc="Retrieving WHO PEN evidence")

        app = ask_medical_rag(query)

        progress(0.90, desc="Formatting verified answer")
        elapsed = time.perf_counter() - t0
        answer = app["answer"]

        det = app.get("deterministic_verification")
        llm_v = app.get("llm_verification")
        verified = bool(
            answer.status == "answer"
            and det and det.get("passed")
            and llm_v and llm_v.get("overall_pass")
        )

        answer_md = answer.answer.strip()
        if answer.safety_note:
            answer_md += f"\\n\\n> **Safety:** {answer.safety_note.strip()}"

        progress(1.0, desc="Ready")
        return (
            _gui_status_html(answer.status, verified, elapsed),
            answer_md,
            _gui_sources(app),
            _gui_evidence(app),
            _gui_dev_trace(app, elapsed),
        )

    except Exception as exc:
        elapsed = time.perf_counter() - t0
        if "_is_rate_limit_error" in globals() and _is_rate_limit_error(exc):
            message = "LLM provider rate-limit/quota reached. Retry when quota resets."
        else:
            message = "Request could not be completed safely. See Developer Trace / Colab logs."

        return (
            f'<div class="pen-status fail">{html.escape(message)} • {elapsed:.2f}s</div>',
            "### Request not completed\\n\\nNo unverified clinical answer was returned.",
            "_No verified citations returned._",
            '<div class="pen-status fail">Evidence display unavailable.</div>',
            {
                "error_type": type(exc).__name__,
                "error": str(exc),
                "traceback_tail": traceback.format_exc()[-4000:],
            },
        )


In [ ]:

# Cell 52 — Launch WHO PEN Clinical Evidence Assistant

with gr.Blocks(
    title="WHO PEN Clinical Evidence Assistant",
    css=GUI_CSS,
    theme=gr.themes.Soft(),
) as demo:

    gr.HTML("""
    <div class="pen-shell">
      <div class="pen-hero">
        <div class="pen-badge">
          <span class="pen-dot"></span>
          Evidence-grounded WHO PEN reference
        </div>
        <div class="pen-title">WHO PEN Clinical Evidence Assistant</div>
        <div class="pen-subtitle">
          Hybrid retrieval · BGE-M3 · BM25 · RRF · Cross-Encoder reranking ·
          parent expansion · citation verification · numeric safety · entailment verification
        </div>
      </div>
    </div>
    """)

    with gr.Column(elem_classes=["pen-shell"]):
        with gr.Group(elem_classes=["pen-card"]):
            question = gr.Textbox(
                label="Clinical reference question",
                placeholder="Example: What intervention is recommended for an AUDIT score of 16–19?",
                lines=3,
            )
            with gr.Row():
                ask_btn = gr.Button("Analyze WHO PEN evidence", variant="primary")
                clear_btn = gr.Button("Clear")

            gr.Markdown(
                "> **Clinical safety:** guideline-reference assistant only. "
                "It does not replace clinician judgment, local protocols, or emergency care."
            )

        status = gr.HTML('<div class="pen-status">Ready for a WHO PEN evidence query.</div>')

        with gr.Tabs():
            with gr.Tab("Answer"):
                answer = gr.Markdown("### Ready\\nSubmit a question to generate a grounded answer.")
            with gr.Tab("Sources"):
                sources = gr.Markdown("_No sources yet._")
            with gr.Tab("Evidence"):
                evidence = gr.HTML('<div class="pen-status">Retrieved evidence will appear here.</div>')
            with gr.Tab("Developer Trace"):
                dev_trace = gr.JSON(label="Retrieval + verification trace", value={})

        gr.Examples(
            examples=[
                ["What are the five steps of the 5A's behavioural counselling intervention?"],
                ["What intervention is listed for an AUDIT score of 16–19?"],
                ["What weekly amount of moderate physical activity does WHO PEN recommend for adults?"],
                ["How should suspected acute stroke be handled according to WHO PEN?"],
                ["What is the purpose of the PEN monitoring system?"],
            ],
            inputs=[question],
        )

    outputs = [status, answer, sources, evidence, dev_trace]

    ask_btn.click(
        fn=run_gui_query,
        inputs=[question],
        outputs=outputs,
        show_progress="full",
    )
    question.submit(
        fn=run_gui_query,
        inputs=[question],
        outputs=outputs,
        show_progress="full",
    )
    clear_btn.click(
        fn=lambda: (
            "",
            '<div class="pen-status">Ready for a WHO PEN evidence query.</div>',
            "### Ready\\nSubmit a question to generate a grounded answer.",
            "_No sources yet._",
            '<div class="pen-status">Retrieved evidence will appear here.</div>',
            {},
        ),
        inputs=[],
        outputs=[question, status, answer, sources, evidence, dev_trace],
    )

demo.queue(default_concurrency_limit=1, max_size=20)

print("Launching WHO PEN Clinical Evidence Assistant...")
demo.launch(share=True, inline=True, show_error=True)



## Final operational outputs

After one complete run, v7 persists the canonical source, parsed document, structured tables, hierarchical chunks, dense/sparse indexes, evaluation outputs, visual QA, and a **secret-safe** run manifest.

Google Drive checkpoint:
`who_pen_medical_rag_v7_gui_checkpoint`

### Recommended run order
1. Run from the top.
2. First environment setup restarts Colab once; reconnect and run from the top again.
3. Reuse the validated checkpoint when available.
4. Configure LLM settings through environment variables / Colab secrets only.
5. Run evaluation cells as needed.
6. Cell 48 is resumable; HTTP 429 keeps the gate **PENDING**.
7. Save the secret-safe manifest and v7 checkpoint.
8. Launch the Gradio GUI.

### Release rule
Do not label the system clinically production-approved merely because the GUI runs. Final release requires the held-out / clinician-reviewed acceptance evidence, zero critical numeric mismatches, valid citations, and no unsupported clinical claims.
